In [1]:
import numpy as np
import h5py
import cv2
from joblib import load

import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
from mpl_toolkits.mplot3d import Axes3D
import mpl_toolkits.mplot3d.axes3d as p3
import matplotlib.animation as animation

%matplotlib widget
import time
%load_ext autoreload
import os
%autoreload 2

from ipywidgets import Video, Image, VBox, Text
from sidecar import Sidecar
from IPython.display import display
from scipy.interpolate import interp1d
from scipy import interpolate
import numpy as np
import os
import cv2
import base64
import warnings

from ipywidgets import interact, interactive, fixed, interact_manual, FloatSlider, Layout, AppLayout
import ipywidgets as widgets
import sklearn

# Our custom-made imports
import sys
sys.path.append('./lib/')
from DataManagement import SplitdataManager, Filepaths, Calibration

# Paths & Parameters (Edit!)

In [2]:
### Paths to files needed and experiment specific parameters.


# experiment folder path
experiment_path = '/media/stephens-group/guest_drive/labelling/complete_shortfin_experiment/FishTank20200130_153857/'
#experiment_path = '/home/thomasreus/Documents/zebrafish_labeling_GUI/FishTank20200130_153857/'
# experiment_path = '/home/thomasreus/Documents/zebrafish_labeling_GUI/OneFish_20200416/'

# tracks folder path
#tracksPath = '/home/liam/Data/results_from_laetitia/FishTank20200130_153857/interpd_updated_old_tracking_on_original_instances_sLEAP.h5'
# tracksPath = '/home/thomasreus/Documents/zebrafish_labeling_GUI/OneFish_20200416/SingleFish_test.h5'
#tracksPath = '/home/thomasreus/Documents/zebrafish_labeling_GUI/FishTank20200130_153857/interpd_updated_old_tracking_on_original_instances_sLEAP.h5'
#tracksPath = '/home/liam/Data/experiments/for_rafael/tracks/FishTank20200130_153857_results_tatsuo_test.h5'
#tracksPath = '/home/liam/fish_analysis/nugget_dump/tracking_20201119_batch1/FishTank20200130_153857_results.h5'
tracksPath = '/home/liam/Data/paired_data_tracks/pair9/FishTank20200130_153857.h5'

# process save location and file
labelsDataSaveFolder = '/home/liam/temp/movie_folder/'
# labelsDataSaveFolder = '/home/thomasreus/Documents/zebrafish_labeling_GUI/GUI_save_folder/Single/'
#labelsDataSaveFolder = '/home/thomasreus/Documents/zebrafish_labeling_GUI/GUI_save_folder/'
fileName = 'FishTank20200130_153857_mouth_wrestling.h5'

# calibration file folder
# calibrationFolder = '20200325_calibration/shared_20200907/'
calibrationFolder = '20200120_calibration/'
calibrationFolderPath = os.path.join(experiment_path, calibrationFolder)

# other parameters
numFish = 2
numBodyPoints = 3
numCams = 3

# Boilerplate Stuff

In [3]:
### Load data, prepare/load save file, 
### create calibration and splitmanager objects 
### and load variable names from splitmanager.

# load image coordinates data
with h5py.File(tracksPath, 'r') as hf:
    #tracks_3D_raw = hf['tracks_3D'][:]
    tracks_imCoords_raw = hf['tracks_imCoords_raw'][:]
    tracks_3D_smooth = hf['tracks_3D_smooth'][:]
totalnumFrames = tracks_3D_smooth.shape[0]

# define the arrays which hold the edited information
tracks_3D = np.copy(tracks_3D_smooth)
tracks_imCoords = np.copy(tracks_imCoords_raw)

# make the save folder if it does not already exist
if not os.path.exists(labelsDataSaveFolder):
    os.makedirs(labelsDataSaveFolder)
    
# save a copy of the image coordinates and 3D data if it doesnt exist
track_save_path = os.path.join(labelsDataSaveFolder, fileName)
if not os.path.exists(track_save_path):
    print('Making a h5 to hold outputs')
    with h5py.File(track_save_path, 'w') as hf:
        hf.create_dataset('tracks_3D', data=tracks_3D)
        hf.create_dataset('tracks_imCoords', data=tracks_imCoords)
        hf.create_dataset('missing_bps', data=np.zeros((3, totalnumFrames, numFish, numBodyPoints),dtype=bool))
        hf.create_dataset('annotated_frames', data=np.zeros((totalnumFrames,), dtype=bool))
    missing_bps = np.zeros((3, totalnumFrames, numFish, numBodyPoints), dtype=bool)
    annotated_frames = np.zeros((totalnumFrames,), dtype=bool)
    tracks_3D_edited = np.copy(tracks_3D)
    tracks_imCoords_edited = np.copy(tracks_imCoords)
else:
    print('found labels file')
    with h5py.File(track_save_path, 'r') as hf:
        tracks_3D_edited = hf['tracks_3D'][:]
        tracks_imCoords_edited = hf['tracks_imCoords'][:]
        annotated_frames = hf['annotated_frames'][:]
        missing_bps = hf['missing_bps'][:]

# set the path to the calibration folder and make the object
#calibrationFolderPath = os.path.join(experiment_path, calibrationFolder)
cal = Calibration(calibrationFolderPath)

# make a splitdata manager
splitman = SplitdataManager(experiment_path)

# gather the names of the splitdata folders with corresponding indices
splitdata_names = [triplet[0].split('/')[-1] for triplet in splitman.splitdata_paths]
splitdata_idxs = [i for i in range(len(splitdata_names))]
splitdata_name_to_idx_dict = dict(zip(splitdata_names, splitdata_idxs))
splitdata_name_to_paths = dict(zip(splitdata_names, splitman.splitdata_paths))
totalnumFrames = splitman.start_stop_frames_for_splitdata[-1, 1]

Making a h5 to hold outputs


FileNotFoundError: [Errno 2] No such file or directory: '/media/stephens-group/guest_drive/labelling/complete_shortfin_experiment/FishTank20200130_153857/20200120_calibration/python_calibration_models/imCoords_to_XYZ.joblib'

# Sidecar Dashboard

In [5]:
### Create Sidecar Dashboard containing:
### Widgets that hold frame specific values and
### create functionality for the labeling GUI.
### Functionalities: 
###    - Changing frame splitdata-wise (splitdata dropbox)
###    - Changing frame index-wise (Global fIdx, Frame slider)
###    - Go to next frame with missing bodypoint (Next NaN)
###    - Change frame specific focus (camIdx, bpIdx, fishIdx)
###    - Lock camIdx (Lock idx)
###    - Save process (Click me to save)
###    - Tick box for missing bodypoints and accepting frames as traning data
###    - Find closest related bodypoint data for missing bodypoints (Repair Frame).

## Widget Initialisation

In [4]:
### Create the widgets.


# make a dropdown menue to choose which splitdata you want
splitdata_chooser = widgets.Dropdown(options=splitdata_name_to_idx_dict, value=0)

# create and initialize widget to hold the splitdata_idx
splitdata_idx = widgets.IntText(value=0, description='splitdata_idx', layout=Layout(width='50%', height='50px'), disabled=True)

# create and initialize widget to hold the filepaths of the movies
xzMovPath =  widgets.Text(value=splitman.splitdata_paths[splitdata_idx.value][0] + '.mp4')
xyMovPath =  widgets.Text(value=splitman.splitdata_paths[splitdata_idx.value][1] + '.mp4')
yzMovPath =  widgets.Text(value=splitman.splitdata_paths[splitdata_idx.value][2] + '.mp4')

# create widgets to hold the global and local frame index
global_index  = widgets.IntText(value=0, description='Global fIdx', layout=Layout(width='70%', height='50px'), disabled=False)
local_index = widgets.IntText(value=0, description='Local fIdx', layout=Layout(width='70%', height='50px'), disabled=True)

# make a Global fIdx slide, and link it to the global frame number
global_index_slider = widgets.IntSlider(value=0, min=0, max=totalnumFrames-1, step=1, continuous_update=False,
                                       description='Frame', layout=Layout(width='90%', height='50px'))
gf_link = widgets.link((global_index_slider, 'value'), (global_index, 'value'))

# button to find next frame with missing data
next_widget = widgets.Button(description="Next NaN")

# --- Make variables to hold fishIdx and bpIdx for when we are clicking --- #
camIdx_widget = widgets.BoundedIntText(value=-1, min=-1, max=numCams-1, description='camIdx')
bpIdx_widget = widgets.BoundedIntText(value=-1, min=-1, max=numBodyPoints-1, description='bpIdx')
fishIdx_widget = widgets.BoundedIntText(value=-1, min=-1, max=numFish-1, description='fishIdx')

# --- Make a widget toggle between drawing/not-drawing the bodypoints --- #
draw_widget = widgets.ToggleButtons(options=['draw', 'blank'],
                                   value='draw',
                                   description='Drawing info',
                                   disabled=False)

# Make the lock_camIdx_widget - the fixed camera view for plotting points
lock_camIdx_widget = widgets.RadioButtons(options=[0, 1, 2],
                                          value=1, 
                                          layout={'width': 'max-content'}, 
                                          description='Lock idx:',
                                          disabled=False)

# make a save button
save_widget = widgets.Button(description="Click Me To Save")

# make a widget to mark bodypoints as missing
missing_bp_widget = widgets.Checkbox(value=False,
                                    description='BodyPoint missing',
                                    disabled=False,
                                    indent=False)

# make a widget to mark frames as annotated
annotated_frame_widget = widgets.Checkbox(value=False,
                                        description='Frame has been verified',
                                        disabled=False,
                                        indent=False)

# button to replace missing data
repair_widget = widgets.Button(description='Repair Frame')

# number of verified frames
verified_number_widget = widgets.IntText(value=len(annotated_frames[annotated_frames]), description='#Verified', layout=Layout(width='70%', height='50px'), disabled=True)

NameError: name 'splitdata_name_to_idx_dict' is not defined

## Widget Functions

In [7]:
### Functionality functions of the dashboard widgets.


def handle_splitdata_name_change(change):
    ''' Make required updates when we select a splitdata**** with the dropdown menu
    '''
    # update splitdata index
    splitdata_idx.value = change.new
    # update moviePaths
    xzMovPath.value = splitman.splitdata_paths[splitdata_idx.value][0] + '.mp4'
    xyMovPath.value = splitman.splitdata_paths[splitdata_idx.value][1] + '.mp4'
    yzMovPath.value = splitman.splitdata_paths[splitdata_idx.value][2] + '.mp4'
    # update the global index given the new splitdata index and the current local idx
    splitdata_start = splitman.start_stop_frames_for_splitdata[splitdata_idx.value][0]
    global_index_slider.value = splitdata_start + local_index.value
    return


def handle_splitdata_idx_box_change(change):
    ''' Make the required changes when we update the "splitdata_idx" 
    '''
    # first, we must make sure that this new change is within the splitdatas available
    numSplitdatas = len(splitman.splitdata_paths)
    if change.new == -1:
        splitdata_idx.value = numSplitdatas - 1
    elif change.new == numSplitdatas:
        splitdata_idx.value = 0
    else:
        splitdata_idx.value = change.new
    return


def handle_local_index_change(change):
    ''' Make the required changes when we update the "Local fIdx"
    '''
    # get the start-stop for the current splitdata index
    splitdata_start_stop = splitman.start_stop_frames_for_splitdata[splitdata_idx.value]
    splitdata_numFrames = splitdata_start_stop[1] - splitdata_start_stop[0]
    # now make sure that the new index is within the range
    local_index.value = np.mod(local_index.value, splitdata_numFrames)
    return


def handle_global_index_slider_change(change):
    ''' Make the required changes when we update the "Global fIdx slider"
    '''
    # get the current splitdata index and local index for this frame
    spIdx, locIdx = splitman.return_splitdata_folder_and_local_idx_for_global_frameIdx(global_index_slider.value, return_splitdataIdx=True)
    local_index.value = locIdx
    splitdata_idx.value = spIdx
    splitdata_chooser.label = splitdata_names[spIdx]
    splitdata_chooser.index = spIdx
    missing_bp_widget.value = bool(missing_bps[camIdx_widget.value, global_index_slider.value,
                                          fishIdx_widget.value, bpIdx_widget.value])
    annotated_frame_widget.value = bool(annotated_frames[change.new])
    graph_index_slider.value = global_index_slider.value
    return


def handle_next_press(p):
    ''' Go to next frame with missing data when next_widget is pressed
    '''
    # get current frame info
    start_frame = global_index_slider.value
    global_fIdx = np.copy(start_frame) + 1
    # loop and check for next frame with nan-value
    while global_fIdx != start_frame:
        if global_fIdx == totalnumFrames:
            global_fIdx = 0
        # go to frames if nan-value found
        if np.any(np.isnan(tracks_imCoords_edited[:, global_fIdx, :, :, :])):
            global_index_slider.value = global_fIdx
            break
        global_fIdx += 1
    # return if no missing instances
    return


# handles to update the missing bodypoint widget
def update_missing_bp_widget_on_camIdx_change(change):
    missing_bp_widget.value = bool(missing_bps[change.new, global_index_slider.value, fishIdx_widget.value, bpIdx_widget.value])
    return
def update_missing_bp_widget_on_bpIdx_change(change):
    missing_bp_widget.value = bool(missing_bps[camIdx_widget.value, global_index_slider.value, fishIdx_widget.value, change.new])
    return
def update_missing_bp_widget_on_fishIdx_change(change):
    missing_bp_widget.value = bool(missing_bps[camIdx_widget.value, global_index_slider.value, change.new, bpIdx_widget.value])
    return


def handle_save_press(p):
    # grab the most current value of the widgets back into the arrays
    # imcoords
    for camIdx in range(numCams):
        for fishIdx in range(numFish):
            for bpIdx in range(numBodyPoints):
                for imCoordIdx in range(2):
                    val = frame_imCoords_widget_list[camIdx][fishIdx][bpIdx][imCoordIdx].value
                    
                    tracks_imCoords_edited[camIdx, global_index_slider.value, fishIdx, bpIdx, imCoordIdx] = val
    # 3D values
    for fishIdx in range(numFish):
        for bpIdx in range(numBodyPoints):
            for dimIdx in range(3):
                val = frame_3D_widget_list[fishIdx][bpIdx][dimIdx].value
                tracks_3D_edited[global_index_slider.value, fishIdx, bpIdx, dimIdx] = val
    # missing bodypoints
    missing_bps[camIdx_widget.value, global_index_slider.value,
                fishIdx_widget.value, bpIdx_widget.value] = missing_bp_widget.value
    # frame annotations
    annotated_frames[global_index_slider.value] = annotated_frame_widget.value
    
    # Now save
    with h5py.File(track_save_path, 'a') as hf:
        hf['tracks_3D'][:] = tracks_3D_edited
        hf['tracks_imCoords'][:] = tracks_imCoords_edited
        hf['annotated_frames'][:] = annotated_frames
        hf['missing_bps'][:] = missing_bps
    return


def handle_missing_bp_press(change):
    if camIdx == -1 or bpIdx == -1 or fishIdx == -1:
        return
    missing_bps[camIdx_widget.value, global_index_slider.value, fishIdx_widget.value, bpIdx_widget.value] = change.new
    return


def handle_annotation_frame_press(change):
    annotated_frames[global_index_slider.value] = change.new
    verified_number_widget.value = len(annotated_frames[annotated_frames])
    return


def handle_repair_press(change):
    ''' Find closest bodypoint data to repair missing data on current frame
    '''
    # start frame
    start_frame = global_index_slider.value
    
    # find closest frame with current fish data and edit when needed
    for fishIdx in range(numFish):
        
        # parameters
        found_frame = start_frame
        gfIdx_min = start_frame - 1
        gfIdx_plus = start_frame + 1

        # move backwards and forwards until frame with complete data is found
        while gfIdx_min != gfIdx_plus:
            # end of range exceptions
            if gfIdx_min == -1:
                gfIdx_min = totalnumFrames - 1
            if gfIdx_plus == totalnumFrames:
                gfIdx_plus = 0
            # gfIdx_min check
            fish_found = np.copy(tracks_imCoords_edited[:, gfIdx_min, fishIdx, :, :])
            if not np.any(np.isnan(fish_found)):
                found_frame = gfIdx_min
                break
            # gfIdx_plus check
            fish_found = np.copy(tracks_imCoords_edited[:, gfIdx_plus, fishIdx, :, :])
            if not np.any(np.isnan(fish_found)):
                found_frame = gfIdx_plus
                break
            # iterate
            gfIdx_min -= 1
            gfIdx_plus += 1
        
        for camIdx in range(numCams):
            for bpIdx in range(numBodyPoints):
                for imCoordIdx in range(2):
                    val1 = frame_imCoords_widget_list[camIdx][fishIdx][bpIdx][imCoordIdx].value
                    if np.isnan(val1):
                        new_val1 = np.copy(tracks_imCoords_edited[camIdx, found_frame, fishIdx, bpIdx, imCoordIdx])
                        frame_imCoords_widget_list[camIdx][fishIdx][bpIdx][imCoordIdx].value = round(float(new_val1), 2)
        
        # update 3D positions in widget
        for bpIdx in range(numBodyPoints):
            update_3D_positions(fishIdx, bpIdx)
                
    # update figures and data
    update_crop_figures(0)
    return


def update_3D_positions(fishIdx, bpIdx):
    '''Updates 3D widget information'''
    
    # get 3 camera view data together
    imageCoord_triplet = []
    for camIdx in range(numCams):
        coords_2D = []
        for imCoordIdx in range(2):
            coord_1D = frame_imCoords_widget_list[camIdx][fishIdx][bpIdx][imCoordIdx].value
            coords_2D.append(coord_1D)
        imageCoord_triplet.append(coords_2D)
    
    # calculate 3D values
    values_3D = cal.compute_XYZ_from_imageCoord_triplet(np.array(imageCoord_triplet))
    
    # change 3D values in widget
    for dimIdx in range(3):
        frame_3D_widget_list[fishIdx][bpIdx][dimIdx].value = round(values_3D[0][dimIdx], 2)
        
    return

## Function-Widget Linking & Sidecar Creation

In [8]:
### Add functionality to widgets and create dashboard Vbox.


# add functionality
splitdata_chooser.observe(handle_splitdata_name_change, names='value')
splitdata_idx.observe(handle_splitdata_idx_box_change, names='value')
local_index.observe(handle_local_index_change, names='value')
global_index_slider.observe(handle_global_index_slider_change, names='value')

next_widget.on_click(handle_next_press)

camIdx_widget.observe(update_missing_bp_widget_on_camIdx_change, names='value')
bpIdx_widget.observe(update_missing_bp_widget_on_bpIdx_change, names='value')
fishIdx_widget.observe(update_missing_bp_widget_on_fishIdx_change, names='value')

save_widget.on_click(handle_save_press)
missing_bp_widget.observe(handle_missing_bp_press, names='value')
annotated_frame_widget.observe(handle_annotation_frame_press, names='value')

repair_widget.on_click(handle_repair_press)


# make Vbox
splitdata_details_box = VBox([splitdata_chooser, 
                              splitdata_idx, 
                              global_index, 
                              local_index, 
                              global_index_slider,
                              next_widget,
                              camIdx_widget,
                              bpIdx_widget,
                              fishIdx_widget,
                              draw_widget,
                              lock_camIdx_widget,
                              save_widget,
                              missing_bp_widget,
                              annotated_frame_widget,
                              verified_number_widget,
                              repair_widget])

# display Vbox
sc = Sidecar(title='Sidecar Output')
with sc:
    display(splitdata_details_box)

# Visual Widgets

In [9]:
### In this section we create the following widgets:
###    - Interactive line-plot of raw data 
###    - 3d and imagecoordinate values of tracking results
###    - Images of 3 camera views including trackresults
### And add interactivity to these widgets with update and change functions.

## Interactive Line-Plot of Raw Data

In [10]:
### In this section we make a widget that contains a plot of the position timeseries, 
### this can be used for identifying by eye places where the data is poor or missing. 
### The POI of the specific global index range can be changed by an interactive slidebar. 

In [11]:
plt.ioff()

fig, (ax1, ax2) = plt.subplots(2, sharex=True, gridspec_kw={'hspace': 0.2})
fig.figsize = (18,10)
#fig.legend(handles, labels, loc='upper center')
fig.suptitle('Trajectory Data Timeseries')
fig.canvas.header_visible = False
fig.canvas.layout.min_height = '400px'
fig.canvas.layout.width = '100%'
ax1.grid(which='both')
ax2.grid(which='both')
ax1.ticklabel_format(useOffset=False)
ax2.ticklabel_format(useOffset=False)
plot_frame_width = 50

#plt.xticks(np.arange(0, totalnumFrames, 100))

# ---- define the info for each plot ----- #
# plot 1 - heads
bpIdx = 0
p1_data = np.copy(tracks_3D_edited[:, :, bpIdx, :])

# plot 2 - pecs
bpIdx = 2
p2_data = np.copy(tracks_3D_edited[:, :, bpIdx, :])

colors = ['lightcoral', 'brown', 'maroon', 
          'forestgreen', 'darkgreen', 'limegreen']


# ----- define the line plots ------ #
for i in range(numFish):
    lines1_x1 = ax1.plot(p1_data[:, i, 0], label='fish{}_head_x'.format(i + 1), color=colors[i*3])
    lines1_y1 = ax1.plot(p1_data[:, i, 1], label='fish{}_head_y'.format(i + 1), color=colors[i*3+1])
    lines1_z1 = ax1.plot(p1_data[:, i, 2], label='fish{}_head_z'.format(i + 1), color=colors[i*3+2])
    lines2_x1 = ax2.plot(p2_data[:, i, 0], label='fish{}_x'.format(i + 1), color=colors[i*3])
    lines2_y1 = ax2.plot(p2_data[:, i, 1], label='fish{}_y'.format(i + 1), color=colors[i*3+1])
    lines2_z1 = ax2.plot(p2_data[:, i, 2], label='fish{}_z'.format(i + 1), color=colors[i*3+2])


ax1.set_title('Heads: frame = {0}'.format(global_index_slider.value))
ax2.set_title('Tails)')

ax1.set_ylim(-1, 43)
ax2.set_ylim(-1, 43)

ax1.set_yticks([i for i in range(0,41,5)], minor=True)
ax2.set_yticks([i for i in range(0,41,5)], minor=True)

ax1.set_ylabel('Head Position (cm)')
ax2.set_ylabel('Tail Position (cm)')

# use any axis here to get the labels
handles, labels = ax2.get_legend_handles_labels()
fig.legend(handles, labels, loc='center right')

if global_index_slider.value < plot_frame_width:
    ax1.set_xlim(0, 2*plot_frame_width)
    ax2.set_xlim(0, 2*plot_frame_width)
elif totalnumFrames - global_index_slider.value < plot_frame_width:
    ax1.set_xlim(totalnumFrames-2*plot_frame_width, totalnumFrames)
    ax2.set_xlim(totalnumFrames-2*plot_frame_width, totalnumFrames)
else:
    ax1.set_xlim(global_index_slider.value-plot_frame_width, global_index_slider.value+plot_frame_width)
    ax2.set_xlim(global_index_slider.value-plot_frame_width, global_index_slider.value+plot_frame_width)


def update_lines(change):
    ax1.set_title('Heads: frame = {0})'.format(change.new))
    ax2.set_title('Tails')
    if change.new < plot_frame_width:
        ax1.set_xlim(0, 2*plot_frame_width)
        ax2.set_xlim(0, 2*plot_frame_width)
    elif totalnumFrames - change.new < plot_frame_width:
        ax1.set_xlim(totalnumFrames-2*plot_frame_width, totalnumFrames)
        ax2.set_xlim(totalnumFrames-2*plot_frame_width, totalnumFrames)
    else:
        ax1.set_xlim(change.new-plot_frame_width, change.new+plot_frame_width)
        ax2.set_xlim(change.new-plot_frame_width, change.new+plot_frame_width)
    fig.canvas.draw()
    fig.canvas.flush_events()

    
# graph index slider
graph_index_slider = widgets.IntSlider(value=0, min=0, max=totalnumFrames-1, step=1, continuous_update=True,
                                       description='Frame', layout=Layout(width='90%', height='50px'))
graph_index_slider.observe(update_lines, names='value')

layout = widgets.Layout(width='100%')
plot_widget = VBox([fig.canvas, graph_index_slider], layout=layout)

plot_widget

## Tracking Result Widgets

In [12]:
### In this section we make a display of widgets for holding the 3D positions and 
### image coordinates

In [13]:
### Grab the image coordinates and 3D coordinats for this frame


# layout parameters
number_layout = widgets.Layout(width='70px')
text_layout = widgets.Layout(width='120px')

# default positions of bodypoints in cropped coords
p1_init = np.array([20, 20])
p2_init = np.array([80, 80])
p3_init = np.array([140, 140])
def_cropped_pos = [p1_init, p2_init, p3_init]

# preallocate list structures to hold the numbers
frame_imCoords_widget_list = [[[[ [] for _ in range(2)] for _ in range(numBodyPoints)] for _ in range(numFish)] for _ in range(numCams)]
frame_3D_widget_list = [ [[[] for _ in range(3)] for _ in range(numBodyPoints)] for _ in range(numFish)]

# fill in the widgets with initial values
for camIdx in range(numCams):
    for fishIdx in range(numFish):
        for bpIdx in range(numBodyPoints):
            for imCoordIdx in range(2):
                wid_val = np.round(np.copy(tracks_imCoords_edited[camIdx, global_index_slider.value, fishIdx, bpIdx, imCoordIdx]),2)
                val_widge_coord = widgets.FloatText(value=wid_val,layout=number_layout)
                frame_imCoords_widget_list[camIdx][fishIdx][bpIdx][imCoordIdx] = val_widge_coord
                
for fishIdx in range(numFish):
    for bpIdx in range(numBodyPoints):
        for dimIdx in range(3):
            wid_val = np.round(np.copy(tracks_3D_edited[global_index_slider.value, fishIdx, bpIdx, dimIdx]), 2)
            val_widge_3D = widgets.FloatText(value=wid_val,layout=number_layout)
            frame_3D_widget_list[fishIdx][bpIdx][dimIdx] = val_widge_3D


In [14]:
### Make widgets to vizualize the 3D positions
### (?) Implement general function for experiments with different number of fish


# layout parameters
number_layout = widgets.Layout(width='80px')
text_layout = widgets.Layout(width='120px')

# widget lists
f_bp_XYZ_boxes = [None]*numFish
f_3D_wid_titles = [None]*numFish
f_3D_wids = [None]*numFish

for fishIdx in range(numFish):
    f_bp_box_list = []
    for bpIdx in range(numBodyPoints):
        f_bp_XYZ_boxes[fishIdx] = widgets.HBox(frame_3D_widget_list[fishIdx][bpIdx][:], layout=widgets.Layout(padding=('0px 30px 0 0')))
        f_bp_box_list.append(f_bp_XYZ_boxes[fishIdx])
    f_3D_wid_titles[fishIdx] = widgets.Text('Fish{} 3D bps'.format(fishIdx+1), layout=text_layout)
    f_3D_wids[fishIdx] = widgets.VBox([f_3D_wid_titles[fishIdx]]+f_bp_box_list)

# Final widget
tracks_3D_frame_widget = widgets.HBox(f_3D_wids, layout=widgets.Layout(padding=('0px 0px 30px 200px')))

In [15]:
### Make widgets to vizualize the image coordinates
### (?) Implement general function for experiments with different number of fish


# layout parameters
number_layout = widgets.Layout(width='80px')
text_layout = widgets.Layout(width='120px')
cam_title_layout = widgets.Layout(width='160px')
title_layout = widgets.Layout(width='480px')
camNames = ['xz', 'xy', 'yz']

# widget lists
bp_imcoords_box = [None]*numFish
cam_widget_titles = [[None]*numCams]*numFish
cam_widgets = [[None]*numCams]*numFish
f_imcoord_wid_title = [None]*numFish
f_imcoord_wid_data = [None]*numFish
f_imcoord_wid = [None]*numFish

for fishIdx in range(numFish):
    cam_bp_imcoord_wid_list = []
    for camIdx in range(3):
        cam_bp_list = []
        for bpIdx in range(numBodyPoints):
            bp_imcoords_box[fishIdx] = widgets.HBox(frame_imCoords_widget_list[camIdx][fishIdx][bpIdx][:])
            cam_bp_list.append(bp_imcoords_box[fishIdx])
        cam_widget_titles[fishIdx][camIdx] = widgets.Text(camNames[camIdx], layout=cam_title_layout)
        cam_widgets[fishIdx][camIdx] = widgets.VBox([cam_widget_titles[fishIdx][camIdx]]+cam_bp_list)
        cam_bp_imcoord_wid_list.append(cam_widgets[fishIdx][camIdx])
    f_imcoord_wid_title[fishIdx] = widgets.Text('Fish{} bodypoint image coordinates'.format(fishIdx + 1), layout=title_layout)
    f_imcoord_wid_data[fishIdx] = widgets.HBox(cam_bp_imcoord_wid_list)
    f_imcoord_wid[fishIdx] = widgets.VBox([f_imcoord_wid_title[fishIdx], f_imcoord_wid_data[fishIdx]])

# final widget
tracks_imCoord_frame_widget = widgets.HBox(f_imcoord_wid)

In [16]:
### Update function for data when changing frame


def update_data(change):
    
    # save the values for the old frame
    for camIdx in range(numCams):
        for fishIdx in range(numFish):
            for bpIdx in range(numBodyPoints):
                for imCoordIdx in range(2):
                    val = frame_imCoords_widget_list[camIdx][fishIdx][bpIdx][imCoordIdx].value
                    tracks_imCoords_edited[camIdx, change.old, fishIdx, bpIdx, imCoordIdx] = val
    for fishIdx in range(numFish):
        for bpIdx in range(numBodyPoints):
            for dimIdx in range(3):
                val = frame_3D_widget_list[fishIdx][bpIdx][dimIdx].value
                tracks_3D_edited[change.old, fishIdx, bpIdx, dimIdx] = val

    
    # update widget for the new frame
    for camIdx in range(numCams):
        for fishIdx in range(numFish):
            for bpIdx in range(numBodyPoints):
                for imCoordIdx in range(2):
                    wid_val = np.round(np.copy(tracks_imCoords_edited[camIdx, change.new, fishIdx, bpIdx, imCoordIdx]),2)
                    frame_imCoords_widget_list[camIdx][fishIdx][bpIdx][imCoordIdx].value = wid_val
    for fishIdx in range(numFish):
        for bpIdx in range(numBodyPoints):
            for dimIdx in range(3):
                wid_val = np.round(np.copy(tracks_3D_edited[change.new, fishIdx, bpIdx, dimIdx]), 2)
                frame_3D_widget_list[fishIdx][bpIdx][dimIdx].value = wid_val
    
    
global_index_slider.observe(update_data, names='value')

In [17]:
# make the final coordinate widget
coordinate_widget = widgets.VBox([tracks_3D_frame_widget, tracks_imCoord_frame_widget])
coordinate_widget

## Loading Images

In [18]:
### In this section we make a widget to hold the the images of the fish tank.
### And we add the following functionalities to the widget:
###    - move bodypoint by clicking near a bodypoint or dragging a bodypoint
###    - move selected bodypoint by clicking crtl+arrowkeys
###    - zoom into figure by using (shift+) scrollwheel
###    - select previous or next bodypoint by pressing (shift+)tab

In [19]:
def draw_skeleton_on_camera_frame(frame, camIdx):
    ''' Draw the image coordinates for the fish on the frame.
    
    --- Inputs ---
    frame: array, an image frame taken from a .mp4
    camIdx: the camera index the frame was taken from
    
    --- Returns ---
    frame: in input frame with the skeleton drawn
    '''
    # check how much info to plot, leaving early if required
    if draw_widget.value == 'blank':
        return
    
    # how bodypoints are joined  
    edges = [[0,1], [1,2]]                  
    
    for fishIdx in range(numFish):

        # draw the bodypoints
        for bpIdx in range(numBodyPoints):
            # get the data
            imCoord_x = frame_imCoords_widget_list[camIdx][fishIdx][bpIdx][0].value
            imCoord_y = frame_imCoords_widget_list[camIdx][fishIdx][bpIdx][1].value
            imCoord = np.array([imCoord_x, imCoord_y])
            #get the color
            fcolor = fish_colors[fishIdx]
            color = ( int (fcolor [ 0 ]), int(fcolor [ 1 ]), int(fcolor [ 2 ]) )
            #color = ( int (no_id_color[ 0 ]), int(no_id_color[ 1 ]), int(no_id_color[ 2 ]))
            if np.all(np.isnan(imCoord)):
                continue
            cv2.circle(frame, (int(imCoord[0]), int(imCoord[1])), radius=2, color=color, thickness=-1)

        # draw the lines
        for edge in edges:
            pt1_bpIdx = edge[0]
            pt2_bpIdx = edge[1]

            pt1_imCoord_x = frame_imCoords_widget_list[camIdx][fishIdx][pt1_bpIdx][0].value
            pt1_imCoord_y = frame_imCoords_widget_list[camIdx][fishIdx][pt1_bpIdx][1].value
            pt1_imCoord = np.array([pt1_imCoord_x, pt1_imCoord_y])

            pt2_imCoord_x = frame_imCoords_widget_list[camIdx][fishIdx][pt2_bpIdx][0].value
            pt2_imCoord_y = frame_imCoords_widget_list[camIdx][fishIdx][pt2_bpIdx][1].value
            pt2_imCoord = np.array([pt2_imCoord_x, pt2_imCoord_y])

            # parse the points for format and emptiness
            if np.all(np.isnan(pt1_imCoord)):
                pt1 =tuple([np.NaN, np.NaN])
            else:
                pt1 = tuple([int(x) for x in pt1_imCoord])
            if np.all(np.isnan(pt2_imCoord)):
                pt2 =tuple([np.NaN, np.NaN])
            else:
                pt2 = tuple([int(x) for x in pt2_imCoord])
            if ~np.all(np.isnan(pt1)) and ~np.all(np.isnan(pt2)):
                cv2.line(frame, pt1=pt1,  pt2=pt2, color=color, thickness=1, lineType=cv2.LINE_AA)
                
    return frame

In [20]:
### Initial creation of figures


# make a list of colors
no_id_color = (255,255,255)           # white
fish_colors = [(255,0,0), (0,0,255)]
#fish_colors = [(255,255,255), (255,255,255)]
edges = [[0,1], [1,2]]                # how bodypoints are joined                  


ROIS = {
    "xz": [182, 278, 962, 1011],
    "xy": [157, 281, 868, 999],
    "yz": [186, 290, 958, 1015],
}


# ----- Make the figure ------#
plt.ioff()

fig1, (f1_ax1, f1_ax2, f1_ax3) = plt.subplots(1, 3, figsize=(9,3))

f_axs = [f1_ax1, f1_ax2, f1_ax3]

f1_ax1.set_title('XZ')
f1_ax2.set_title('XY')
f1_ax3.set_title('YZ')

for fax in f_axs:
    fax.get_xaxis().set_visible(False)
    fax.get_yaxis().set_visible(False)
    
fig1.canvas.header_visible = False
fig1.canvas.layout.min_height = '400px'
fig1.canvas.layout.width = '100%'

plt.subplots_adjust(top = 1, bottom = 0, right = 1, left = 0, 
           hspace = 0, wspace = 0)



#  -------------- get the initial frame data -------------------#
video_paths = [xzMovPath, xyMovPath, yzMovPath]
caps = [cv2.VideoCapture(video_path.value) for video_path in video_paths]
[cap.set(1,local_index.value) for cap in caps] #go to current frame
# grab the frame from each camera
if caps[0].isOpened() and caps[1].isOpened() and caps[2].isOpened():  
    # grab the 3 images
    ret_xz, frame_xz = caps[0].read()
    if not ret_xz:
        raise TypeError('xz movie for splitdata_idx={0} not opening'.format(splitdata_idx.value))
    ret_xy, frame_xy = caps[1].read()
    if not ret_xy:
        raise TypeError('xy movie for splitdata_idx={0} not opening'.format(splitdata_idx.value))
    ret_yz, frame_yz = caps[2].read()
    if not ret_yz:
        raise TypeError('yz movie for splitdata_idx={0} not opening'.format(splitdata_idx.value))
    frames = [frame_xz, frame_xy, frame_yz]
# close everything before we finish
[caps[capIdx].release() for capIdx in range(3)];


# draw the info
for camIdx in range(3):
    frame = frames[camIdx]
    frame = draw_skeleton_on_camera_frame(frame, camIdx)
                          
                
# ------------ plot the images -------------#

panel1 = f1_ax1.imshow(frames[0])
panel2 = f1_ax2.imshow(frames[1])
panel3 = f1_ax3.imshow(frames[2])




In [21]:
### Update function for figure
### (?) This could maybe be (rewritten a little and) called 
###     in other functions to do the drawing


def update_crop_figures(change):
    
    caps = [cv2.VideoCapture(video_path.value) for video_path in video_paths]
    [cap.set(1,local_index.value) for cap in caps] #go to current frame
    # grab the frame from each camera
    if caps[0].isOpened() and caps[1].isOpened() and caps[2].isOpened():  
        # grab the 3 images
        ret_xz, frame_xz = caps[0].read()
        if not ret_xz:
            raise TypeError('xz movie for splitdata_idx={0} not opening'.format(splitdata_idx.value))
        ret_xy, frame_xy = caps[1].read()
        if not ret_xy:
            raise TypeError('xy movie for splitdata_idx={0} not opening'.format(splitdata_idx.value))
        ret_yz, frame_yz = caps[2].read()
        if not ret_yz:
            raise TypeError('yz movie for splitdata_idx={0} not opening'.format(splitdata_idx.value))
        frames = [frame_xz, frame_xy, frame_yz]
    # release the caps
    [caps[capIdx].release() for capIdx in range(3)];
    
    # --------------- Draw the skeletons --------------------#
    # draw the info
    for camIdx in range(3):
        frame = frames[camIdx]
        frame = draw_skeleton_on_camera_frame(frame, camIdx)
    
    # now update the figures    
    panel1.set_data(frames[0])
    panel2.set_data(frames[1])
    panel3.set_data(frames[2])
    
    fig1.canvas.draw()
    fig1.canvas.flush_events()
    return

global_index_slider.observe(update_crop_figures, names='value')
draw_widget.observe(update_crop_figures, names='value')

In [22]:
### Functionality functions


def click_down_event_handler(event):
    # find the fishIdx and camIdx image we clicked on
    for fishIdx in range(numFish):
        for camIdx in range(numCams):
            comp_ax = f_axs[camIdx]
            if event.inaxes == comp_ax:
                camIdx_widget.value = camIdx
            
    # find the closest image coordinate, wihtin the theshold
    find_idx_of_closest_point(event.xdata, event.ydata, tolerance=20)

    
def find_idx_of_closest_point(x, y, tolerance=15):
    ''' Return the bodypoint corresponding to this click
    '''
    camIdx = camIdx_widget.value
    
    distances = [[] for _ in range(numFish)]
    
    for fishIdx in range(numFish):
        for bpIdx in range(numBodyPoints):
            xp = frame_imCoords_widget_list[camIdx][fishIdx][bpIdx][0].value 
            yp = frame_imCoords_widget_list[camIdx][fishIdx][bpIdx][1].value 
            dist = (((x-xp)**2) + ((y-yp)**2))**0.5
            distances[fishIdx].append(dist)
            
    # the fish with the lowest cost is the fishIdx
    best_fishIdx = np.nanargmin([np.min(fish_distances) for fish_distances in distances])
    best_bpIdx = np.nanargmin(distances[best_fishIdx])
    min_dist = np.nanmin(distances[best_fishIdx])
    
    if min_dist < tolerance:
        bpIdx_widget.value = best_bpIdx
        fishIdx_widget.value = best_fishIdx
    else:
        bpIdx_widget.value = -1
        fishIdx_widget.value = -1

         
def click_release_event_handler(event):
    ''' When you release the mouse, redraw the appropriate figure
    '''
    # do nothing if we dont have an active bpIdx and fishIdx
    if camIdx_widget.value == -1 or bpIdx_widget.value == -1:
        return
    
    # do nothing if cam is locked
    if camIdx_widget.value == lock_camIdx_widget.value:
        return
    
    # -------------------------------------------------------#
    #      Update the image you clicked on
    # -------------------------------------------------------#
    # update the image coordinates using the mouse click
    frame_imCoords_widget_list[camIdx_widget.value][fishIdx_widget.value][bpIdx_widget.value][0].value = event.xdata
    frame_imCoords_widget_list[camIdx_widget.value][fishIdx_widget.value][bpIdx_widget.value][1].value = event.ydata
    
    # grab the frame we want to redraw
    cap = cv2.VideoCapture(video_paths[camIdx_widget.value].value)
    cap.set(1,local_index.value) #go to current frame
    # grab the frame from each camera
    if cap.isOpened():
        # grab the 3 images
        ret, frame = cap.read()
        if not ret:
            raise TypeError('movie for splitdata_idx={0} not opening'.format(splitdata_idx.value))
    # release the caps
    cap.release();
    
    # draw the info
    frame = draw_skeleton_on_camera_frame(frame, camIdx_widget.value)
            
    # now update the figures
    if camIdx_widget.value == 0:
        panel1.set_data(frame)
    elif camIdx_widget.value == 1:
        panel2.set_data(frame)
    elif camIdx_widget.value == 2:
        panel3.set_data(frame)
        
    # -------------------------------------------------------#
    #     Update the other image
    # -------------------------------------------------------#
    
    # decide on "other_camIdx" - the other camera to update
    if lock_camIdx_widget.value == 0:
        if camIdx_widget.value == 1:
            other_camIdx = 2
        elif camIdx_widget.value == 2:
            other_camIdx = 1
            
    if lock_camIdx_widget.value == 1:
        if camIdx_widget.value == 0:
            other_camIdx = 2
        elif camIdx_widget.value == 2:
            other_camIdx = 0
            
    if lock_camIdx_widget.value == 2:
        if camIdx_widget.value == 0:
            other_camIdx = 1
        elif camIdx_widget.value == 1:
            other_camIdx = 0
            
    # grab the frame we want to redraw
    cap = cv2.VideoCapture(video_paths[other_camIdx].value)
    cap.set(1,local_index.value) #go to current frame
    # grab the frame from each camera
    if cap.isOpened():
        # grab the 3 images
        ret, frame = cap.read()
        if not ret:
            raise TypeError('movie for splitdata_idx={0} not opening'.format(splitdata_idx.value))
    # release the caps
    cap.release();

    # get the image coordinates of the other point
    clicked_point_imCoord_x = frame_imCoords_widget_list[camIdx_widget.value][fishIdx_widget.value][bpIdx_widget.value][0].value
    clicked_point_imCoord_y = frame_imCoords_widget_list[camIdx_widget.value][fishIdx_widget.value][bpIdx_widget.value][1].value
    clicked_point_imcoords = np.array([clicked_point_imCoord_x, clicked_point_imCoord_y])
    fixed_point_imCoord_x = frame_imCoords_widget_list[lock_camIdx_widget.value][fishIdx_widget.value][bpIdx_widget.value][0].value
    fixed_point_imCoord_y = frame_imCoords_widget_list[lock_camIdx_widget.value][fishIdx_widget.value][bpIdx_widget.value][1].value
    fixed_point_imcoords = np.array([fixed_point_imCoord_x, fixed_point_imCoord_y])
    
    if lock_camIdx_widget.value == 0:
        if camIdx_widget.value == 1:
            other_camIdx = 2
            other_point_imcoords = cal.compute_YZ_imcoords_from_XZ_XY(fixed_point_imcoords, clicked_point_imcoords)
        elif camIdx_widget.value == 2:
            other_camIdx = 1
            other_point_imcoords = cal.compute_XY_imcoords_from_XZ_YZ(fixed_point_imcoords, clicked_point_imcoords)
            
    if lock_camIdx_widget.value == 1:
        if camIdx_widget.value == 0:
            other_camIdx = 2
            other_point_imcoords = cal.compute_YZ_imcoords_from_XZ_XY(clicked_point_imcoords, fixed_point_imcoords)
        elif camIdx_widget.value == 2:
            other_camIdx = 0
            other_point_imcoords = cal.compute_XZ_imcoords_from_XY_YZ(fixed_point_imcoords, clicked_point_imcoords)
            
    if lock_camIdx_widget.value == 2:
        if camIdx_widget.value == 0:
            other_camIdx = 1
            other_point_imcoords = cal.compute_XY_imcoords_from_XZ_YZ(clicked_point_imcoords, fixed_point_imcoords)
        elif camIdx_widget.value == 1:
            other_camIdx = 0
            other_point_imcoords = cal.compute_XZ_imcoords_from_XY_YZ(clicked_point_imcoords, fixed_point_imcoords)

    # update the image coordinates using the mouse click
    frame_imCoords_widget_list[other_camIdx][fishIdx_widget.value][bpIdx_widget.value][0].value = other_point_imcoords[0][0]
    frame_imCoords_widget_list[other_camIdx][fishIdx_widget.value][bpIdx_widget.value][1].value = other_point_imcoords[0][1]
    
    # update 3D positions in widget
    update_3D_positions(fishIdx_widget.value, bpIdx_widget.value)
    
    # draw the info
    frame = draw_skeleton_on_camera_frame(frame, other_camIdx)
            
    # now update the figure
    if other_camIdx == 0:
        panel1.set_data(frame)
    elif other_camIdx == 1:
        panel2.set_data(frame)
    elif other_camIdx == 2:
        panel3.set_data(frame)
        
    # ----------------------------------------------------------------------#
    fig1.canvas.draw()
    fig1.canvas.flush_events()
    return


def key_press_event_handler(event):
    '''
    (shift+)tab to move between bodypoints
    key interaction for moving bodypoint indicators (ctrl + arrowkeys)
                                                    (t, y, u, i)
    '''
    # return if no camera selected
    if camIdx_widget.value == -1:
        return
    # check if tab is pressed
    if event.key == 'tab' or event.key == 'shift+tab':
        fishIdx = fishIdx_widget.value
        bpIdx = bpIdx_widget.value
        camIdx = camIdx_widget.value
        if event.key == 'tab':
            bpIdx += 1
            if bpIdx == numBodyPoints:
                bpIdx = 0
                fishIdx += 1
            if fishIdx == numFish:
                fishIdx = 0
        else:
            bpIdx -= 1
            if bpIdx == -1:
                bpIdx = numBodyPoints - 1
                fishIdx -= 1
            if fishIdx == -1:
                fishIdx = numFish - 1
        bpIdx = max([0, bpIdx])
        fishIdx = max([0, fishIdx])
        bpIdx_widget.value = bpIdx
        fishIdx_widget.value = fishIdx
        # get right ax
        axes = [f1_ax1, f1_ax2, f1_ax3]
        ax = axes[camIdx]
        # current axes ranges
        cur_xlim = ax.get_xlim()
        cur_ylim = ax.get_ylim()
        cur_xrange = (cur_xlim[1] - cur_xlim[0])*.5
        cur_yrange = (cur_ylim[1] - cur_ylim[0])*.5
        # center and crop all cameraviews
        for camIdx, ax in enumerate(axes):
            # make sure home button resets to full image
            ax.figure.canvas.toolbar.push_current()
            imCoord_x = frame_imCoords_widget_list[camIdx][fishIdx][bpIdx][0].value
            imCoord_y = frame_imCoords_widget_list[camIdx][fishIdx][bpIdx][1].value
            if np.isnan(imCoord_x) or np.isnan(imCoord_y):
                continue
            # set new limits
            ax.set_xlim([imCoord_x - cur_xrange,
                         imCoord_x + cur_xrange])
            ax.set_ylim([imCoord_y - cur_yrange,
                         imCoord_y + cur_yrange])
        fig1.canvas.draw()
        fig1.canvas.flush_events()
        return
    # return if no bodypoint or fish is selected
    if bpIdx_widget.value == -1 or fishIdx_widget.value == -1:
        return
    # do nothing if cam is locked
    if camIdx_widget.value == lock_camIdx_widget.value:
        return
    key_actions = {'ctrl+right': [0, 1], 'ctrl+left': [0, -1], 'ctrl+up': [1, -1], 'ctrl+down': [1, 1]}
    #key_actions = {'t': [1, 0], 'y': [1, 1], 'u': [1, -1], 'i': [0, 1]}
    try:
        key_action = key_actions[event.key]
        frame_imCoords_widget_list[camIdx_widget.value][fishIdx_widget.value][bpIdx_widget.value][key_action[0]].value += key_action[1]
    except:
        return
    # grab the frame we want to redraw
    cap = cv2.VideoCapture(video_paths[camIdx_widget.value].value)
    cap.set(1,local_index.value) #go to current frame
    # grab the frame from each camera
    if cap.isOpened():
        # grab the 3 images
        ret, frame = cap.read()
        if not ret:
            raise TypeError('movie for splitdata_idx={0} not opening'.format(splitdata_idx.value))
    # release the caps
    cap.release();
    # --------------- Draw the skeletons --------------------#
    frame = draw_skeleton_on_camera_frame(frame, camIdx_widget.value)

    # now update the figure
    if camIdx_widget.value == 0:
        panel1.set_data(frame)
    elif camIdx_widget.value == 1:
        panel2.set_data(frame)
    elif camIdx_widget.value == 2:
        panel3.set_data(frame)
        
    # -------------------------------------------------------#
    #     Update the other image
    # -------------------------------------------------------#
    
    # decide on "other_camIdx" - the other camera to update
    if lock_camIdx_widget.value == 0:
        if camIdx_widget.value == 1:
            other_camIdx = 2
        elif camIdx_widget.value == 2:
            other_camIdx = 1
            
    if lock_camIdx_widget.value == 1:
        if camIdx_widget.value == 0:
            other_camIdx = 2
        elif camIdx_widget.value == 2:
            other_camIdx = 0
            
    if lock_camIdx_widget.value == 2:
        if camIdx_widget.value == 0:
            other_camIdx = 1
        elif camIdx_widget.value == 1:
            other_camIdx = 0
            
    # grab the frame we want to redraw
    cap = cv2.VideoCapture(video_paths[other_camIdx].value)
    cap.set(1,local_index.value) #go to current frame
    # grab the frame from each camera
    if cap.isOpened():
        # grab the 3 images
        ret, frame = cap.read()
        if not ret:
            raise TypeError('movie for splitdata_idx={0} not opening'.format(splitdata_idx.value))
    # release the caps
    cap.release();
     
    # get the image coordinates of the other point
    clicked_point_imCoord_x = frame_imCoords_widget_list[camIdx_widget.value][fishIdx_widget.value][bpIdx_widget.value][0].value
    clicked_point_imCoord_y = frame_imCoords_widget_list[camIdx_widget.value][fishIdx_widget.value][bpIdx_widget.value][1].value
    clicked_point_imcoords = np.array([clicked_point_imCoord_x, clicked_point_imCoord_y])
    fixed_point_imCoord_x = frame_imCoords_widget_list[lock_camIdx_widget.value][fishIdx_widget.value][bpIdx_widget.value][0].value
    fixed_point_imCoord_y = frame_imCoords_widget_list[lock_camIdx_widget.value][fishIdx_widget.value][bpIdx_widget.value][1].value
    fixed_point_imcoords = np.array([fixed_point_imCoord_x, fixed_point_imCoord_y])
    
    if lock_camIdx_widget.value == 0:
        if camIdx_widget.value == 1:
            other_camIdx = 2
            other_point_imcoords = cal.compute_YZ_imcoords_from_XZ_XY(fixed_point_imcoords, clicked_point_imcoords)
        elif camIdx_widget.value == 2:
            other_camIdx = 1
            other_point_imcoords = cal.compute_XY_imcoords_from_XZ_YZ(fixed_point_imcoords, clicked_point_imcoords)
            
    if lock_camIdx_widget.value == 1:
        if camIdx_widget.value == 0:
            other_camIdx = 2
            other_point_imcoords = cal.compute_YZ_imcoords_from_XZ_XY(clicked_point_imcoords, fixed_point_imcoords)
        elif camIdx_widget.value == 2:
            other_camIdx = 0
            other_point_imcoords = cal.compute_XZ_imcoords_from_XY_YZ(fixed_point_imcoords, clicked_point_imcoords)
            
    if lock_camIdx_widget.value == 2:
        if camIdx_widget.value == 0:
            other_camIdx = 1
            other_point_imcoords = cal.compute_XY_imcoords_from_XZ_YZ(clicked_point_imcoords, fixed_point_imcoords)
        elif camIdx_widget.value == 1:
            other_camIdx = 0
            other_point_imcoords = cal.compute_XZ_imcoords_from_XY_YZ(clicked_point_imcoords, fixed_point_imcoords)

    # update the image coordinates using the mouse click
    frame_imCoords_widget_list[other_camIdx][fishIdx_widget.value][bpIdx_widget.value][0].value = other_point_imcoords[0][0]
    frame_imCoords_widget_list[other_camIdx][fishIdx_widget.value][bpIdx_widget.value][1].value = other_point_imcoords[0][1]
    
    # update 3D positions in widget
    update_3D_positions(fishIdx_widget.value, bpIdx_widget.value)
    
    # draw the points for this view
    frame = draw_skeleton_on_camera_frame(frame, other_camIdx)

    # now update the figure
    if other_camIdx == 0:
        panel1.set_data(frame)
    elif other_camIdx == 1:
        panel2.set_data(frame)
    elif other_camIdx == 2:
        panel3.set_data(frame)    
        
    fig1.canvas.draw()
    fig1.canvas.flush_events()
    return


def scroll_event_handler(event):
    '''scrolling event handler to resize figures
    '''
    base_scale = 1.5
    # select image from cursor location
    c_width = fig1.canvas._width
    if event.x < c_width / 3:
        ax = f1_ax1
    elif event.x > c_width / 3 and event.x < 2 * c_width / 3:
        ax = f1_ax2
    elif event.x > 2 * c_width / 3:
        ax = f1_ax3
    else:
        return
    # get current size and register event
    cur_xlim = ax.get_xlim()
    cur_ylim = ax.get_ylim()
    cur_xrange = (cur_xlim[1] - cur_xlim[0])*.5
    cur_yrange = (cur_ylim[1] - cur_ylim[0])*.5
    xdata = event.xdata 
    ydata = event.ydata
    if xdata == None or ydata == None:
        return
    if event.button == 'up':
        scale_factor = 1/base_scale
    elif event.button == 'down':
        scale_factor = base_scale
    else:
        scale_factor = 1
        print(event.button)
    # make sure home button resets to full image
    ax.figure.canvas.toolbar.push_current()
    # set new limits
    ax.set_xlim([xdata - cur_xrange*scale_factor,
                 xdata + cur_xrange*scale_factor])
    ax.set_ylim([ydata - cur_yrange*scale_factor,
                 ydata + cur_yrange*scale_factor])
    fig1.canvas.draw()
    fig1.canvas.flush_events()
    return

# connect functions to canvas
fig1.canvas.mpl_connect('button_press_event', click_down_event_handler)
fig1.canvas.mpl_connect('button_release_event', click_release_event_handler)
fig1.canvas.mpl_connect('key_press_event', key_press_event_handler)
fig1.canvas.mpl_connect('scroll_event', scroll_event_handler)

12

In [23]:
# Make the final widget
image_widget = widgets.VBox([fig1.canvas])

# Combined View (Labeling Zone)

In [24]:
# In this section we give an example of sticking all the widgets together

In [25]:
display(plot_widget, coordinate_widget, image_widget)

In [26]:
# 149868 - cool, face off



In [27]:
plt.ion()
plt.figure()
plt.imshow(frames[0])


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

# getting Nice still shots

In [44]:
# 149868 - cool, face off

# 95766 - cool, little bit of z difference

# 94614, attack


# global_index.value = 94614

# xz_center = [700,690]
# xy_center = [540,590]
# yz_center = [700,600]

In [28]:
import io


def get_img_from_fig(fig, dpi=180):
    # define a function which returns an image as numpy array from figure
    # Thanks: https://stackoverflow.com/a/58641662/4978308
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=dpi)
    buf.seek(0)
    img_arr = np.frombuffer(buf.getvalue(), dtype=np.uint8)
    buf.close()
    img = cv2.imdecode(img_arr, 1)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    return img

# img = get_img_from_fig(fig1, dpi=300)



In [76]:
xzim.shape

(1024, 1280, 3)

In [77]:
xz_center[0]-cropSize[0]

600

In [119]:
# set frame
chose_frame = 94614

cropSize = [70,70]

xz_center = [700,690-5]
xy_center = [540-5,590]
yz_center = [700,600+15]


# ---- #

global_index.value = chose_frame

xzim =np.copy(panel1._A.data)
xyim =np.copy(panel2._A.data)
yzim =np.copy(panel3._A.data)

#chose_frame_img = get_img_from_fig(fig1, dpi=300)

xzim_crop = xzim[ (xz_center[0]-cropSize[0]):(xz_center[0]+cropSize[0]), (xz_center[1]-cropSize[1]):(xz_center[1]+cropSize[1]) ]
xyim_crop = xyim[ (xy_center[0]-cropSize[0]):(xy_center[0]+cropSize[0]), (xy_center[1]-cropSize[1]):(xy_center[1]+cropSize[1]) ]
yzim_crop = yzim[ (yz_center[0]-cropSize[0]):(yz_center[0]+cropSize[0]), (yz_center[1]-cropSize[1]):(yz_center[1]+cropSize[1]) ]

copy_frames = [xzim_crop, xyim_crop, yzim_crop]

In [123]:
# plt.ion()

# for frame in copy_frames:
#     plt.figure()
#     plt.imshow(frame)

In [125]:
#final_panel = np.hstack([xzim_crop, xyim_crop, yzim_crop])

#plt.figure()
#plt.imshow(final_panel)

# plt.imsave('/home/liam/Dropbox/fight_3panel.pdf', final_panel)

In [126]:
plt.imsave('/home/liam/Dropbox/transfer/fight_3panel_xz.pdf', xzim_crop)
plt.imsave('/home/liam/Dropbox/transfer/fight_3panel_xy.pdf', xyim_crop)
plt.imsave('/home/liam/Dropbox/transfer/fight_3panel_yz.pdf', yzim_crop)

In [127]:
plt.imsave('/home/liam/Dropbox/transfer/fight_3panel_xz_full.pdf', xzim)
plt.imsave('/home/liam/Dropbox/transfer/fight_3panel_xy_full.pdf', xyim)
plt.imsave('/home/liam/Dropbox/transfer/fight_3panel_yz_full.pdf', yzim)

# Movie with panels

In [26]:
f0

122040

In [27]:
fE

122106

In [29]:
# some annotations
offset_for_modelData = 50000
fight_bout_continuing = 153500 - offset_for_modelData 
fight_stop_frame = 178000 - offset_for_modelData 
one_froze_bottom_other_moves = 184900 - offset_for_modelData 
fish_frozen_at_bottom_facing = 199000 - offset_for_modelData 
pair_facing_at_bottom = 226000 - offset_for_modelData
lined_up_in_straight_line = 235400 - offset_for_modelData 
chaser_chasee_at_bottom = 241900 - offset_for_modelData 

In [30]:
t3D = np.copy(tracks_3D)
t0 = time.time()

# pick a start and stop frame
#startFrame = 153500
#stopFrame = startFrame + 1000

startFrame = 122040
stopFrame = 122100

title = ' Mouth wrestling? : '

# set other parameters for the movie
movPath = '/home/liam/temp/fishmovies4/mouth_wrestling_panels.mp4'
fps = 1
fourcc = cv2.VideoWriter_fourcc('m','p','4','v')
# THIS NEED TO MATCH THE SHAPE OF THE FRAMES YOU MAKE
nrows = 500
ncols = int(500*(2176/780))
resolution = (ncols,nrows) 
out = cv2.VideoWriter(movPath, fourcc, fps, resolution)


# loop over the frames
for fIdx in range(startFrame, stopFrame):
    if np.mod(fIdx, 2) == 0:
        continue
    # 
    if np.mod(fIdx, 11) == 0:
        print(fIdx)
    #print(fIdx)
    
    # update the frame
    global_index_slider.value = fIdx

    # get the frames
    caps = [cv2.VideoCapture(video_path.value) for video_path in video_paths]
    [cap.set(1,local_index.value) for cap in caps] #go to current frame
    # grab the frame from each camera
    if caps[0].isOpened() and caps[1].isOpened() and caps[2].isOpened():  
        # grab the 3 images
        ret_xz, frame_xz = caps[0].read()
        if not ret_xz:
            raise TypeError('xz movie for splitdata_idx={0} not opening'.format(splitdata_idx.value))
        ret_xy, frame_xy = caps[1].read()
        if not ret_xy:
            raise TypeError('xy movie for splitdata_idx={0} not opening'.format(splitdata_idx.value))
        ret_yz, frame_yz = caps[2].read()
        if not ret_yz:
            raise TypeError('yz movie for splitdata_idx={0} not opening'.format(splitdata_idx.value))
        frames = [frame_xz, frame_xy, frame_yz]
    # release the caps
    [caps[capIdx].release() for capIdx in range(3)];
    
    # Draw the skeletons 
    for camIdx in range(3):
        frame = frames[camIdx]
        frame = draw_skeleton_on_camera_frame(frame, camIdx)
        if camIdx == 0:
            frame = frame[ROIS['xz'][0]:ROIS['xz'][2], ROIS['xz'][1]:ROIS['xz'][3]]
        elif camIdx == 1:
            frame = frame[ROIS['xy'][0]-34:ROIS['xy'][2]+35, ROIS['xy'][1]:ROIS['xy'][3]]
        elif camIdx == 2:
            frame = frame[ROIS['yz'][0]-4:ROIS['yz'][2]+4, ROIS['yz'][1]:ROIS['yz'][3]]
    
    # stick the 3 panels together
    trip_panel = np.hstack(frames)
    
    # resize the panel
    nrows = 500
    ncols = int(500*(2176/780))
    movie_panel = cv2.resize(trip_panel, (ncols,nrows), interpolation=cv2.INTER_AREA)
    movie_panel = cv2.cvtColor(movie_panel, cv2.COLOR_BGR2RGB)
    
    # draw the frame number
    font                   = cv2.FONT_HERSHEY_SIMPLEX
    bottomLeftCornerOfText = (550,30)
    fontScale              = 1
    fontColor              = (0,0,0)
    lineType               = 2
    cv2.putText(movie_panel, title+str(fIdx).zfill(6), 
        bottomLeftCornerOfText, 
        font, 
        fontScale,
        fontColor,
        lineType)
    
    # write the frame
    out.write(movie_panel)

# close the writer
out.release()

print()
print(time.time()-t0)

122045


KeyboardInterrupt: 

# Single panel movie

In [38]:
# some annotations
offset_for_modelData = 50000
fight_bout_continuing = 153500 - offset_for_modelData 
fight_stop_frame = 178000 - offset_for_modelData 
one_froze_bottom_other_moves = 184900 - offset_for_modelData 
fish_frozen_at_bottom_facing = 199000 - offset_for_modelData 
pair_facing_at_bottom = 226000 - offset_for_modelData
lined_up_in_straight_line = 235400 - offset_for_modelData 
chaser_chasee_at_bottom = 241900 - offset_for_modelData 

In [52]:
t3D = np.copy(tracks_3D)
t0 = time.time()

# pick a start and stop frame
#startFrame = 153500
#stopFrame = startFrame + 1000

startFrame = fight_bout_continuing
stopFrame = startFrame + 300

# set other parameters for the movie
movPath = '/home/liam/temp/single_panel_movie.mp4'
fps = 25
fourcc = cv2.VideoWriter_fourcc('m','p','4','v')
# THIS NEED TO MATCH THE SHAPE OF THE FRAMES YOU MAKE
nrows = 780
ncols = 718
resolution = (ncols,nrows) 
out = cv2.VideoWriter(movPath, fourcc, fps, resolution)


# loop over the frames
for fIdx in range(startFrame, stopFrame):

    if np.mod(fIdx, 10) == 0:
        print(fIdx)
    
    # update the frame
    global_index_slider.value = fIdx

    # get the xy frame
    camIdx = 1
    cap = cv2.VideoCapture(video_paths[camIdx].value)
    cap.set(1,local_index.value)
    # grab the frame from each camera
    if cap.isOpened():
        ret_xy, frame_xy = cap.read()
        if not ret_xy:
            raise TypeError('xy movie for splitdata_idx={0} not opening'.format(splitdata_idx.value))
        frame = np.copy(frame_xy)
    # release the caps
    cap.release()
    
    # Draw the skeletons 
    frame = draw_skeleton_on_camera_frame_no_color(frame, camIdx)
    if camIdx == 0:
        frame = frame[ROIS['xz'][0]:ROIS['xz'][2], ROIS['xz'][1]:ROIS['xz'][3]]
    elif camIdx == 1:
        frame = frame[ROIS['xy'][0]-34:ROIS['xy'][2]+35, ROIS['xy'][1]:ROIS['xy'][3]]
    elif camIdx == 2:
        frame = frame[ROIS['yz'][0]-4:ROIS['yz'][2]+4, ROIS['yz'][1]:ROIS['yz'][3]]
    
    # convert to color
    movie_panel = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # write the frame
    out.write(movie_panel)

# close the writer
out.release()

print()
print(time.time()-t0)

103500
103510
103520
103530
103540
103550
103560
103570
103580
103590
103600
103610
103620
103630
103640
103650
103660
103670
103680
103690
103700
103710
103720
103730
103740
103750
103760
103770
103780
103790

300.3682668209076


In [49]:
frame.shape

(780, 718, 3)

# Movie with 3 panels - no colors

In [35]:
def draw_skeleton_on_camera_frame_no_color(frame, camIdx):
    ''' Draw the image coordinates for the fish on the frame.
    
    --- Inputs ---
    frame: array, an image frame taken from a .mp4
    camIdx: the camera index the frame was taken from
    
    --- Returns ---
    frame: in input frame with the skeleton drawn
    '''
    # check how much info to plot, leaving early if required
    if draw_widget.value == 'blank':
        return
    
    # how bodypoints are joined  
    edges = [[0,1], [1,2]]                  
    
    for fishIdx in range(numFish):

        # draw the bodypoints
        for bpIdx in range(numBodyPoints):
            # get the data
            imCoord_x = frame_imCoords_widget_list[camIdx][fishIdx][bpIdx][0].value
            imCoord_y = frame_imCoords_widget_list[camIdx][fishIdx][bpIdx][1].value
            imCoord = np.array([imCoord_x, imCoord_y])
            #get the color
            #fcolor = fish_colors[fishIdx]
            #color = ( int (fcolor [ 0 ]), int(fcolor [ 1 ]), int(fcolor [ 2 ]) )
            color = (0,255,0)
            #color = ( int (no_id_color[ 0 ]), int(no_id_color[ 1 ]), int(no_id_color[ 2 ]))
            if np.all(np.isnan(imCoord)):
                continue
            cv2.circle(frame, (int(imCoord[0]), int(imCoord[1])), radius=2, color=color, thickness=-1)

        # draw the lines
        for edge in edges:
            pt1_bpIdx = edge[0]
            pt2_bpIdx = edge[1]

            pt1_imCoord_x = frame_imCoords_widget_list[camIdx][fishIdx][pt1_bpIdx][0].value
            pt1_imCoord_y = frame_imCoords_widget_list[camIdx][fishIdx][pt1_bpIdx][1].value
            pt1_imCoord = np.array([pt1_imCoord_x, pt1_imCoord_y])

            pt2_imCoord_x = frame_imCoords_widget_list[camIdx][fishIdx][pt2_bpIdx][0].value
            pt2_imCoord_y = frame_imCoords_widget_list[camIdx][fishIdx][pt2_bpIdx][1].value
            pt2_imCoord = np.array([pt2_imCoord_x, pt2_imCoord_y])

            # parse the points for format and emptiness
            if np.all(np.isnan(pt1_imCoord)):
                pt1 =tuple([np.NaN, np.NaN])
            else:
                pt1 = tuple([int(x) for x in pt1_imCoord])
            if np.all(np.isnan(pt2_imCoord)):
                pt2 =tuple([np.NaN, np.NaN])
            else:
                pt2 = tuple([int(x) for x in pt2_imCoord])
            if ~np.all(np.isnan(pt1)) and ~np.all(np.isnan(pt2)):
                cv2.line(frame, pt1=pt1,  pt2=pt2, color=color, thickness=1, lineType=cv2.LINE_AA)
                
    return frame

In [37]:
t3D = np.copy(tracks_3D)
t0 = time.time()

# pick a start and stop frame
#startFrame = 153500
#stopFrame = startFrame + 1000

startFrame = 18000
stopFrame = 18000 + 200

#title = ' Mouth wrestling? : '

# set other parameters for the movie
movPath = '/home/liam/temp/no_color_movie.mp4'
fps = 10
fourcc = cv2.VideoWriter_fourcc('m','p','4','v')
# THIS NEED TO MATCH THE SHAPE OF THE FRAMES YOU MAKE
nrows = 500
ncols = int(500*(2176/780))
resolution = (ncols,nrows) 
out = cv2.VideoWriter(movPath, fourcc, fps, resolution)


# loop over the frames
for fIdx in range(startFrame, stopFrame):
    if np.mod(fIdx, 2) == 0:
        continue
    # 
    if np.mod(fIdx, 11) == 0:
        print(fIdx)
    #print(fIdx)
    
    # update the frame
    global_index_slider.value = fIdx

    # get the frames
    caps = [cv2.VideoCapture(video_path.value) for video_path in video_paths]
    [cap.set(1,local_index.value) for cap in caps] #go to current frame
    # grab the frame from each camera
    if caps[0].isOpened() and caps[1].isOpened() and caps[2].isOpened():  
        # grab the 3 images
        ret_xz, frame_xz = caps[0].read()
        if not ret_xz:
            raise TypeError('xz movie for splitdata_idx={0} not opening'.format(splitdata_idx.value))
        ret_xy, frame_xy = caps[1].read()
        if not ret_xy:
            raise TypeError('xy movie for splitdata_idx={0} not opening'.format(splitdata_idx.value))
        ret_yz, frame_yz = caps[2].read()
        if not ret_yz:
            raise TypeError('yz movie for splitdata_idx={0} not opening'.format(splitdata_idx.value))
        frames = [frame_xz, frame_xy, frame_yz]
    # release the caps
    [caps[capIdx].release() for capIdx in range(3)];
    
    # Draw the skeletons 
    for camIdx in range(3):
        frame = frames[camIdx]
        frame = draw_skeleton_on_camera_frame_no_color(frame, camIdx)
        if camIdx == 0:
            frame = frame[ROIS['xz'][0]:ROIS['xz'][2], ROIS['xz'][1]:ROIS['xz'][3]]
        elif camIdx == 1:
            frame = frame[ROIS['xy'][0]-34:ROIS['xy'][2]+35, ROIS['xy'][1]:ROIS['xy'][3]]
        elif camIdx == 2:
            frame = frame[ROIS['yz'][0]-4:ROIS['yz'][2]+4, ROIS['yz'][1]:ROIS['yz'][3]]
    
    # stick the 3 panels together
    trip_panel = np.hstack(frames)
    
    # resize the panel
    nrows = 500
    ncols = int(500*(2176/780))
    movie_panel = cv2.resize(trip_panel, (ncols,nrows), interpolation=cv2.INTER_AREA)
    movie_panel = cv2.cvtColor(movie_panel, cv2.COLOR_BGR2RGB)
    
#     # draw the frame number
#     font                   = cv2.FONT_HERSHEY_SIMPLEX
#     bottomLeftCornerOfText = (550,30)
#     fontScale              = 1
#     fontColor              = (0,0,0)
#     lineType               = 2
#     cv2.putText(movie_panel, title+str(fIdx).zfill(6), 
#         bottomLeftCornerOfText, 
#         font, 
#         fontScale,
#         fontColor,
#         lineType)
    
    # write the frame
    out.write(movie_panel)

# close the writer
out.release()

print()
print(time.time()-t0)

18007
18029
18051
18073
18095
18117
18139
18161
18183

124.28566718101501


# 3D animation

In [401]:

def animate_scatters(iteration, data, scatters, lines, scatter_projections, lines_projections, text, all_timestamps):
    """
    Update the data held by the scatter plot and therefore animates it.

    Args:
        iteration (int): Current iteration of the animation
        data (list): List of the data positions at each iteration.
        scatters (list): List of all the scatters (One per element)

    Returns:
        list: List of scatters (One per element) with new coordinates
    """
    #if np.mod(iteration, 100) == 0:
        #print(iteration)
    # parse the input shapes
    numFrames, numFish, numBodyPoints, _ = data.shape
 
    # update the scatters
    for fishIdx in range(numFish):
        for bpIdx in range(numBodyPoints):
            scatters[fishIdx][bpIdx]._offsets3d = (data[iteration,fishIdx,bpIdx,0:1],
                                                   data[iteration,fishIdx,bpIdx,1:2],
                                                   data[iteration,fishIdx,bpIdx,2:])
            
    # update the lines
    for fishIdx,line in enumerate(lines):
        line.set_data(data[iteration, fishIdx, :, :2].swapaxes(0,1))
        line.set_3d_properties(data[iteration, fishIdx, :, 2])
        
        
    # update the text
    text.set_text(all_timestamps[iteration])
        
    # -------------- #
    for projectionIdx in range(3):
        
        # XZ => Y=0
        if projectionIdx == 0:
            projectionData = np.copy(data[iteration])
            projectionData[:, :, 1] = 0
        # XY => Z=0
        elif projectionIdx == 1:
            projectionData = np.copy(data[iteration])
            projectionData[:, :, 2] = 0
        # YZ => X=0
        elif projectionIdx == 2:
            projectionData = np.copy(data[iteration])
            projectionData[:, :, 0] = 0
            
        proj_scatters = scatter_projections[projectionIdx]
        proj_lines = lines_projections[projectionIdx]
        
        # update the scatters
        for fishIdx in range(numFish):
            for bpIdx in range(numBodyPoints):
                proj_scatters[fishIdx][bpIdx]._offsets3d = (projectionData[fishIdx,bpIdx,0:1],
                                                            projectionData[fishIdx,bpIdx,1:2],
                                                            projectionData[fishIdx,bpIdx,2:])
        # update the lines
        for fishIdx,line in enumerate(proj_lines):
            line.set_data(projectionData[fishIdx, :, :2].swapaxes(0,1))
            line.set_3d_properties(projectionData[fishIdx, :, 2])

    return




def make_movie(data, f0, fE, savePath='/home/liam/temp/testanimation.mp4'):
    """
    Creates the 3D figure and animates it with the input data.

    Args:
        data (list): List of the data positions at each iteration.
        save (bool): Whether to save the recording of the animation. (Default to False).
    """
    data = tracks_3D[f0:fE]
    
    # parse the input shapes
    numFrames, numFish, numBodyPoints, _ = data.shape
    fish_colors = ['red', 'blue']

    # Attaching 3D axis to the figure
    fig = plt.figure()
    fig.tight_layout()
    ax = p3.Axes3D(fig, [0.,0.,1.,1.])

    # Initialize scatters (list over fish, list over bps)
    symbols = ['o', 's', 'x']
    sizes=12
    
    # Main scatters
    scatters = []
    for fishIdx in range(numFish):
        col = fish_colors[fishIdx]
        fish_scatters = []
        for bpIdx in range(numBodyPoints):
            fish_scatters.append(ax.scatter(data[0,fishIdx,bpIdx,0:1], data[0,fishIdx,bpIdx,1:2], data[0,fishIdx,bpIdx,2:], 
                                            c=col, s=sizes, marker=symbols[bpIdx])) 
        scatters.append(fish_scatters)
        
    # Main lines
    lines = []
    for fishIdx in range(numFish):
        col = fish_colors[fishIdx]
        line = ax.plot(data[0, fishIdx, :, 0], data[0, fishIdx, :, 1], data[0, fishIdx, :, 2], c=col)[0]
        lines.append(line)
        
    # ----------------
    # projections
    symbols = ['x', 'o', 's']
    sizes=5

    scatter_projections = []
    lines_projections = []
    for projectionIdx in range(3):
        proj_scatters = []
        proj_lines = []
        for fishIdx in range(numFish):
            col = fish_colors[fishIdx]
            # scatter plots
            fish_scatters = []
            for bpIdx in range(numBodyPoints):
                #print(symbols[bpIdx])
                fish_scatters.append(ax.scatter(data[0,fishIdx,bpIdx,0:1], data[0,fishIdx,bpIdx,1:2], data[0,fishIdx,bpIdx,2:], 
                                                c=col, s=sizes, marker=symbols[bpIdx], alpha=0.1))
            # lines
            line = ax.plot(data[0, fishIdx, :, 0], data[0, fishIdx, :, 1], data[0, fishIdx, :, 2], c=col, alpha=0.1)[0]
            proj_lines.append(line)
            proj_scatters.append(fish_scatters)
        
        scatter_projections.append(proj_scatters)
        lines_projections.append(proj_lines)
    # ----------------
    
    # the text
    all_timestamps = ['frame: '+ str(i).zfill(7) for i in range(f0, fE)]
    time_text = ax.text(14, 0, 43,'', fontsize=15)

    # Setting the axes properties
    ax.set_xlim3d([0, 40])
    ax.set_xlabel('X')

    ax.set_ylim3d([0, 40])
    ax.set_ylabel('Y')

    ax.set_zlim3d([0, 40])
    ax.set_zlabel('Z')

    # Provide starting angle for the view
    elevation_angle = 10
    azimuthal_angle = 45
    ax.view_init(elevation_angle, azimuthal_angle)

    ani = animation.FuncAnimation(fig, animate_scatters, numFrames, fargs=(data, scatters, lines, 
                                                                           scatter_projections, lines_projections,
                                                                           time_text, all_timestamps),
                                       interval=10, blit=False, repeat=True)

    Writer = animation.writers['ffmpeg']
    writer = Writer(fps=100, metadata=dict(artist='LOS'), bitrate=3500)
    ani.save(savePath, writer=writer)
    return

In [403]:
t0 = time.time()

saveFolder = '/home/liam/temp/fishmovies3/'

if not os.path.exists(saveFolder):
    os.makedirs(saveFolder)
    

st_sps = splitman.start_stop_frames_for_splitdata

for i in range(1):
#for i in range(st_sps.shape[0]):
    savePath = os.path.join(saveFolder, splitdata_names[i]+'.mp4')
    print(savePath)

    #f0, fE = st_sps[i]
    f0, fE = 0, 10
    n_movFrames = fE - f0
    movie_tracks = tracks_3D[f0:fE]

    main(movie_tracks, savePath=savePath)
    print(time.time()-t0)

print('Finished!')
tE = time.time()
print(tE-t0)
    


/home/liam/temp/fishmovies3/splitdata0009.mp4


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

1.3716187477111816
Finished!
1.3723597526550293


# interacting with the cross-fish-bp-distance configuration

In [417]:
def unit_vector(vector):
    """ Returns the unit vector of the vector.  """
    return vector / np.linalg.norm(vector)

def angle_between(v1, v2):
    """ Returns the angle in radians between vectors 'v1' and 'v2'::

            >>> angle_between((1, 0, 0), (0, 1, 0))
            1.5707963267948966
            >>> angle_between((1, 0, 0), (1, 0, 0))
            0.0
            >>> angle_between((1, 0, 0), (-1, 0, 0))
            3.141592653589793
    """
    v1_u = unit_vector(v1)
    v2_u = unit_vector(v2)
    return np.arccos(np.clip(np.dot(v1_u, v2_u), -1.0, 1.0))

In [11]:
np.set_printoptions(suppress=True)

In [12]:
# load the data

tsp = '/home/liam/fish_analysis/nugget_dump/20201208/c_bp_dist_config.h5'
with h5py.File(tsp, 'r') as hf:
    fish_bp_distances_cm = hf['fish_bp_distances_cm'][:]
    dist_in_pca_basis_entire = hf['dist_in_pca_basis_entire'][:]

In [354]:
def rotation_matrix_from_vectors(vec1, vec2):
    # thanks: https://stackoverflow.com/a/59204638
    """ Find the rotation matrix that aligns vec1 to vec2
    :param vec1: A 3d "source" vector
    :param vec2: A 3d "destination" vector
    :return mat: A transform matrix (3x3) which when applied to vec1, aligns it with vec2.
    """
    a, b = (vec1 / np.linalg.norm(vec1)).reshape(3), (vec2 / np.linalg.norm(vec2)).reshape(3)
    v = np.cross(a, b)
    c = np.dot(a, b)
    s = np.linalg.norm(v)
    kmat = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    rotation_matrix = np.eye(3) + kmat + kmat.dot(kmat) * ((1 - c) / (s ** 2))
    return rotation_matrix


# def rotation_matrix_from_vectors2(vec1, vec2):
#     # thanks: https://stackoverflow.com/a/59204638
#     """ Find the rotation matrix that aligns vec1 to vec2
#     :param vec1: A 3d "source" vector
#     :param vec2: A 3d "destination" vector
#     :return mat: A transform matrix (3x3) which when applied to vec1, aligns it with vec2.
#     """
#     a, b = (vec1 / np.linalg.norm(vec1)).reshape(3), (vec2 / np.linalg.norm(vec2)).reshape(3)
#     v = np.cross(b, a)
#     c = np.dot(a, b)
#     s = np.linalg.norm(v)
#     kmat = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
#     rotation_matrix = np.eye(3) + kmat + kmat.dot(kmat) * ((1 - c) / (s ** 2))
#     return rotation_matrix


def transform_frame_positions_to_pec1_basis(frame_positions):
    ''' Transform the lab coordinate bodypoint positions for a frame into
        the coordinate system centered on the pec of fish1, with x_hat
        defined using the x&y components of the fish1 pec -> head vector
        
    -- inputs --
    frame_positions: (numFish=2, numBodyPoints=3, 3), lab coords
    frame_positions_prime: (numFish=2, numBodyPoints=3, 3), new coords
        
    -- NOTES ---
    HARD CODED FOR 2 FISH
    '''
    # parse shapes
    numFish, numBodyPoints, _ = frame_positions.shape

    # define the vectors for this frame
    fishIdx = 0
    f1_H = frame_positions[fishIdx, 0]
    f1_P = frame_positions[fishIdx, 1]
    f1_T = frame_positions[fishIdx, 2]
    fishIdx = 1
    f2_H = frame_positions[fishIdx, 0]
    f2_P = frame_positions[fishIdx, 1]
    f2_T = frame_positions[fishIdx, 2]

    # define the old basis
    x_lab = np.array([1,0,0])
    y_lab = np.array([0,1,0])
    z_lab = np.array([0,0,1])

    # find the rotation matrix for the x direction
    x_hat = (f1_H - f1_P) / (np.linalg.norm(f1_H - f1_P))
    R = rotation_matrix_from_vectors(x_lab, x_hat)
    #print(R)
    #print('x_hat', x_hat)
    #print('angle', angle_between(x_lab, x_hat))

    # find y_hat and z_hat
    y_hat = np.dot(R, y_lab)
    z_hat = np.dot(R, z_lab)

    # determine the Q matrix
    Q = np.array([x_hat, y_hat, z_hat]).T

    # now write all these points in these coordinates
    f1_H_hat = np.linalg.solve(Q, f1_H - f1_P)
    f1_P_hat = np.linalg.solve(Q, f1_P - f1_P)
    f1_T_hat = np.linalg.solve(Q, f1_T - f1_P)
    f1_hat = np.vstack((f1_H_hat, f1_P_hat, f1_T_hat))

    f2_H_hat = np.linalg.solve(Q, f2_H - f1_P)
    f2_P_hat = np.linalg.solve(Q, f2_P - f1_P)
    f2_T_hat = np.linalg.solve(Q, f2_T - f1_P)
    f2_hat = np.vstack((f2_H_hat, f2_P_hat, f2_T_hat))

    frame_positions_prime = np.stack((f1_hat, f2_hat), axis=0)
    return frame_positions_prime




In [456]:
for fIdx in range(129000, 129010):
    print(fIdx)
    print()
    print(lab_coord_data[fIdx])
    print()
    print(transform_frame_positions_to_pec1_basis(lab_coord_data[fIdx]))
    print()
    print()
    print()

129000

[[[18.47728195 25.97450943  1.40061181]
  [19.32332629 25.82717892  1.4972738 ]
  [21.2393329  26.23774391  1.59649142]]

 [[10.10963052 21.83684148  0.7044676 ]
  [ 9.86966191 22.73697166  0.89086208]
  [ 9.21361976 24.68317776  1.24237464]]]

[[[ 0.86419954  0.          0.        ]
  [-0.          0.          0.        ]
  [-1.81685861 -0.39402685  0.62710182]]

 [[ 8.42852911  2.38133635 -4.97319401]
  [ 8.79606397  2.24625889 -4.10761075]
  [ 9.7307999   1.93084844 -2.27227548]]]



129001

[[[18.29233331 25.90863119  1.40921535]
  [19.13873021 25.73841411  1.47926636]
  [21.0891022  26.05592485  1.58735291]]

 [[10.13052971 21.73189478  0.67663851]
  [ 9.89449407 22.63262807  0.86535612]
  [ 9.25248587 24.53955571  1.21316266]]]

[[[ 0.86618054 -0.          0.        ]
  [-0.         -0.          0.        ]
  [-1.85217119 -0.52740256  0.45580155]]

 [[ 8.08002446  3.9794477  -4.08916687]
  [ 8.47241398  3.53492226 -3.34682246]
  [ 9.44636938  2.58579757 -1.82363824]]]





In [446]:
f =129003
lab_coord_data[f]

array([[[17.95388949, 25.77271015,  1.41916122],
        [18.76005248, 25.71247887,  1.45578755],
        [20.76948163, 25.80378955,  1.59776767]],

       [[10.17798821, 21.57618807,  0.6208174 ],
        [ 9.95097881, 22.46492522,  0.80710218],
        [ 9.30679663, 24.2775522 ,  1.15808283]]])

In [447]:
f =129004
lab_coord_data[f]

array([[[17.79110741, 25.68279651,  1.41812072],
        [18.55351695, 25.74628837,  1.44557897],
        [20.59851047, 25.77301267,  1.59877698]],

       [[10.19649168, 21.45815612,  0.60851449],
        [ 9.9539235 , 22.42382954,  0.77920838],
        [ 9.32425764, 24.23746121,  1.12212735]]])

In [448]:
transform_frame_positions_to_pec1_basis(lab_coord_data[129003])

x_hat [-0.99619866  0.07442951 -0.04526021]
angle 3.054371610880249


array([[[ 0.80923919, -0.        , -0.        ],
        [-0.        ,  0.        ,  0.        ],
        [-2.00142045, -0.06549777,  0.23733455]],

       [[ 8.27936973,  1.79040354, -4.4389634 ],
        [ 8.56323317,  1.56595079, -3.57575339],
        [ 9.32399406,  1.09599145, -1.8367434 ]]])

In [449]:
transform_frame_positions_to_pec1_basis(lab_coord_data[129004])

x_hat [-0.99590909 -0.08293721 -0.03586775]
angle 3.051108417717598


array([[[ 0.7655413 , -0.        , -0.        ],
        [-0.        , -0.        , -0.        ],
        [-2.04433895,  0.03999475,  0.15893707]],

       [[ 8.70850682,  2.83764835,  2.24461296],
        [ 8.86387   ,  2.03536656,  1.65072147],
        [ 9.32824268,  0.4979176 ,  0.54440278]]])

In [441]:
transform_frame_positions_to_pec1_basis(lab_coord_data[129004])

[[-0.99590909  0.08293721  0.03586775]
 [-0.08293721 -0.68143215 -0.72716686]
 [-0.03586775 -0.72716686  0.68552306]]
angle 3.051108417717598


array([[[ 0.7655413 , -0.        , -0.        ],
        [-0.        , -0.        , -0.        ],
        [-2.04433895,  0.03999475,  0.15893707]],

       [[ 8.70850682,  2.83764835,  2.24461296],
        [ 8.86387   ,  2.03536656,  1.65072147],
        [ 9.32824268,  0.4979176 ,  0.54440278]]])

In [ ]:
transform_frame_positions_to_pec1_basis(lab_coord_data[129004])

In [431]:
transform_frame_positions_to_pec1_basis2(lab_coord_data[f])

[[-0.99619866  0.07442951 -0.04526021]
 [-0.07442951 -0.45731382  0.88618526]
 [ 0.04526021  0.88618526  0.46111517]]
angle 3.054371610880249


array([[[ 0.80923919, -0.        , -0.        ],
        [-0.        ,  0.        ,  0.        ],
        [-2.03301477, -0.0678583 ,  0.23876997]],

       [[ 8.95542383,  1.84091392, -4.46967851],
        [ 9.12644887,  1.60803062, -3.60134192],
        [ 9.6572099 ,  1.12088718, -1.85188237]]])

In [432]:
f = 129004
lab_coord_data[f]

array([[[17.79110741, 25.68279651,  1.41812072],
        [18.55351695, 25.74628837,  1.44557897],
        [20.59851047, 25.77301267,  1.59877698]],

       [[10.19649168, 21.45815612,  0.60851449],
        [ 9.9539235 , 22.42382954,  0.77920838],
        [ 9.32425764, 24.23746121,  1.12212735]]])

In [434]:
transform_frame_positions_to_pec1_basis(lab_coord_data[f])

[[-0.99590909  0.08293721  0.03586775]
 [-0.08293721 -0.68143215 -0.72716686]
 [-0.03586775 -0.72716686  0.68552306]]
angle 3.051108417717598


array([[[ 0.7655413 , -0.        , -0.        ],
        [-0.        , -0.        , -0.        ],
        [-2.04433895,  0.03999475,  0.15893707]],

       [[ 8.70850682,  2.83764835,  2.24461296],
        [ 8.86387   ,  2.03536656,  1.65072147],
        [ 9.32824268,  0.4979176 ,  0.54440278]]])

In [435]:
transform_frame_positions_to_pec1_basis2(lab_coord_data[f])

[[-0.99590909 -0.08293721 -0.03586775]
 [ 0.08293721 -0.68143215 -0.72716686]
 [ 0.03586775 -0.72716686  0.68552306]]
angle 3.051108417717598


array([[[ 0.7655413 , -0.        , -0.        ],
        [-0.        , -0.        , -0.        ],
        [-2.06259889,  0.0415154 ,  0.1595947 ]],

       [[ 8.06893506,  2.89091054,  2.26764719],
        [ 8.40216528,  2.07381636,  1.66734981],
        [ 9.20508431,  0.50817397,  0.54883834]]])

In [19]:
#### PLOT A SINGLE FRAME OF THE FISH1 PEC COORD SYSTEM

f0 = 100
fE = 101

data = tracks_3D[f0:fE]
data_p = np.copy(data)
for i in range(data.shape[0]):
    data_p[i] = transform_frame_positions_to_pec1_basis(data[i])
    
# parse the input shapes
numFrames, numFish, numBodyPoints, _ = data.shape
fish_colors = ['red', 'blue']


#### 
data = np.copy(data_p)
###

# set the limits for the plotting.
# This is need to defining the projections too
limval = 10
x_l, x_h = -limval, limval
y_l, y_h = -limval, limval
z_l, z_h = -limval, limval



# Attaching 3D axis to the figure
fig = plt.figure()
fig.tight_layout()
ax = p3.Axes3D(fig, [0.,0.,1.,1.])

## data_lims

# Initialize scatters (list over fish, list over bps)
symbols = ['o', 's', 'x']
sizes=12

# Main scatters
scatters = []
for fishIdx in range(numFish):
    col = fish_colors[fishIdx]
    fish_scatters = []
    for bpIdx in range(numBodyPoints):
        fish_scatters.append(ax.scatter(data[0,fishIdx,bpIdx,0:1], data[0,fishIdx,bpIdx,1:2], data[0,fishIdx,bpIdx,2:], 
                                        c=col, s=sizes, marker=symbols[bpIdx])) 
    scatters.append(fish_scatters)

# Main lines
lines = []
for fishIdx in range(numFish):
    col = fish_colors[fishIdx]
    line = ax.plot(data[0, fishIdx, :, 0], data[0, fishIdx, :, 1], data[0, fishIdx, :, 2], c=col)[0]
    lines.append(line)

# ----------------
# projections
symbols = ['o', 's', 'x']
sizes=5

scatter_projections = []
lines_projections = []
for projectionIdx in range(3):
    
    # THE PATTERN IS NOT THE SAME FOR THE THREE CAMERA
    # IT DEPENDS ON THE VIEW THAT YOU WANT
    
    # XZ => Y=Y+L
    if projectionIdx == 0:
        init_projectionData = np.copy(data[0])
        init_projectionData[:, :, 1] = -y_l
    # XY => Z=Z_L
    elif projectionIdx == 1:
        init_projectionData = np.copy(data[0])
        init_projectionData[:, :, 2] = z_l
    # YZ => X=X_L
    elif projectionIdx == 2:
        init_projectionData = np.copy(data[0])
        init_projectionData[:, :, 0] = x_l
    
    proj_scatters = []
    proj_lines = []
    for fishIdx in range(numFish):
        col = fish_colors[fishIdx]
        # scatter plots
        fish_scatters = []
        for bpIdx in range(numBodyPoints):
            fish_scatters.append(ax.scatter(init_projectionData[fishIdx,bpIdx,0:1], 
                                            init_projectionData[fishIdx,bpIdx,1:2], 
                                            init_projectionData[fishIdx,bpIdx,2:], 
                                            c=col, s=sizes, marker=symbols[bpIdx], alpha=0.1))
        # lines
        line = ax.plot(init_projectionData[fishIdx, :, 0], 
                       init_projectionData[fishIdx, :, 1], 
                       init_projectionData[fishIdx, :, 2], c=col, alpha=0.1)[0]
        proj_lines.append(line)
        proj_scatters.append(fish_scatters)

    scatter_projections.append(proj_scatters)
    lines_projections.append(proj_lines)
# ----------------

# the text
all_timestamps = ['frame: '+ str(i).zfill(7) for i in range(f0, fE)]
time_text = ax.text(14, 0, 43,'', fontsize=15)

# Setting the axes properties
ax.set_xlim3d([x_l, x_h])
ax.set_xlabel('X')
ax.set_ylim3d([y_l, y_h])
ax.set_ylabel('Y')
ax.set_zlim3d([z_l, z_h])
ax.set_zlabel('Z')

# Provide starting angle for the view
elevation_angle = 25
azimuthal_angle = -45
ax.view_init(elevation_angle, azimuthal_angle)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

# Movie of fish1pec coord system

In [20]:
def animate_scatters_f1p(iteration, data, scatters, lines, scatter_projections, lines_projections, text, all_timestamps, all_lims):
    """
    Update the data held by the scatter plot and therefore animates it.

    Args:
        iteration (int): Current iteration of the animation
        data (list): List of the data positions at each iteration.
        scatters (list): List of all the scatters (One per element)

    Returns:
        list: List of scatters (One per element) with new coordinates
    """
    #if np.mod(iteration, 100) == 0:
        #print(iteration)
    # parse the input shapes
    numFrames, numFish, numBodyPoints, _ = data.shape
    #print('i :', iteration)
 
    # update the scatters
    for fishIdx in range(numFish):
        for bpIdx in range(numBodyPoints):
            scatters[fishIdx][bpIdx]._offsets3d = (data[iteration,fishIdx,bpIdx,0:1],
                                                   data[iteration,fishIdx,bpIdx,1:2],
                                                   data[iteration,fishIdx,bpIdx,2:])
            
    # update the lines
    for fishIdx,line in enumerate(lines):
        line.set_data(data[iteration, fishIdx, :, :2].swapaxes(0,1))
        line.set_3d_properties(data[iteration, fishIdx, :, 2])
        
        
    # update the text
    text.set_text(all_timestamps[iteration])
    
    # unpack the axes limits, for defining the projection data
    x_l, x_h = all_lims[0][0], all_lims[0][1]
    y_l, y_h = all_lims[1][0], all_lims[1][1]
    z_l, z_h = all_lims[2][0], all_lims[2][1]
        
    # -------------- #
    for projectionIdx in range(3):
        
        # XZ => Y
        if projectionIdx == 0:
            projectionData = np.copy(data[iteration])
            projectionData[:, :, 1] = y_h
        # XY => Z
        elif projectionIdx == 1:
            projectionData = np.copy(data[iteration])
            projectionData[:, :, 2] = z_l
        # YZ => X
        elif projectionIdx == 2:
            projectionData = np.copy(data[iteration])
            projectionData[:, :, 0] = x_l
            
        proj_scatters = scatter_projections[projectionIdx]
        proj_lines = lines_projections[projectionIdx]
        
        # update the scatters
        for fishIdx in range(numFish):
            for bpIdx in range(numBodyPoints):
                proj_scatters[fishIdx][bpIdx]._offsets3d = (projectionData[fishIdx,bpIdx,0:1],
                                                            projectionData[fishIdx,bpIdx,1:2],
                                                            projectionData[fishIdx,bpIdx,2:])
        # update the lines
        for fishIdx,line in enumerate(proj_lines):
            line.set_data(projectionData[fishIdx, :, :2].swapaxes(0,1))
            line.set_3d_properties(projectionData[fishIdx, :, 2])

    return




def make_f1p_movie(lab_data, f0, fE, savePath='/home/liam/temp/testanimation.mp4'):
    """
    Creates the 3D figure and animates it with the input data.

    Args:
        data (list): List of the data positions at each iteration.
        save (bool): Whether to save the recording of the animation. (Default to False).
    """

    # change coordinate systems
    data = np.copy(lab_data[f0:fE])
    #print(data.shape)
    for i in range(data.shape[0]):
        data[i] = transform_frame_positions_to_pec1_basis(data[i])
        #print('a :', data[i])

    # parse the input shapes
    numFrames, numFish, numBodyPoints, _ = data.shape
    #print(numFrames)
    fish_colors = ['red', 'blue']

    # set the limits for the plotting.
    # This is need to defining the projections too
    limval = 10
    x_l, x_h = -limval, limval
    y_l, y_h = -limval, limval
    z_l, z_h = -limval, limval
    all_lims = [[x_l, x_h], [y_l, y_h], [z_l, z_h]]

    # Attaching 3D axis to the figure
    fig = plt.figure()
    fig.tight_layout()
    ax = p3.Axes3D(fig, [0.,0.,1.,1.])

    # Initialize scatters (list over fish, list over bps)
    symbols = ['o', 's', 'x']
    sizes=12

    # Main scatters
    scatters = []
    for fishIdx in range(numFish):
        col = fish_colors[fishIdx]
        fish_scatters = []
        for bpIdx in range(numBodyPoints):
            fish_scatters.append(ax.scatter(data[0,fishIdx,bpIdx,0:1], data[0,fishIdx,bpIdx,1:2], data[0,fishIdx,bpIdx,2:], 
                                            c=col, s=sizes, marker=symbols[bpIdx])) 
        scatters.append(fish_scatters)

    # Main lines
    lines = []
    for fishIdx in range(numFish):
        col = fish_colors[fishIdx]
        line = ax.plot(data[0, fishIdx, :, 0], data[0, fishIdx, :, 1], data[0, fishIdx, :, 2], c=col)[0]
        lines.append(line)

    # ----------------
    # projections
    symbols = ['o', 's', 'x']
    sizes=5

    scatter_projections = []
    lines_projections = []
    for projectionIdx in range(3):

        # THE PATTERN IS NOT THE SAME FOR THE THREE CAMERA
        # IT DEPENDS ON THE VIEW THAT YOU WANT

        # XZ => Y=
        if projectionIdx == 0:
            init_projectionData = np.copy(data[0])
            init_projectionData[:, :, 1] = y_h
        # XY => Z=
        elif projectionIdx == 1:
            init_projectionData = np.copy(data[0])
            init_projectionData[:, :, 2] = z_l
        # YZ => X=X
        elif projectionIdx == 2:
            init_projectionData = np.copy(data[0])
            init_projectionData[:, :, 0] = x_l

        proj_scatters = []
        proj_lines = []
        for fishIdx in range(numFish):
            col = fish_colors[fishIdx]
            # scatter plots
            fish_scatters = []
            for bpIdx in range(numBodyPoints):
                fish_scatters.append(ax.scatter(init_projectionData[fishIdx,bpIdx,0:1], 
                                                init_projectionData[fishIdx,bpIdx,1:2], 
                                                init_projectionData[fishIdx,bpIdx,2:], 
                                                c=col, s=sizes, marker=symbols[bpIdx], alpha=0.1))
            # lines
            line = ax.plot(init_projectionData[fishIdx, :, 0], 
                           init_projectionData[fishIdx, :, 1], 
                           init_projectionData[fishIdx, :, 2], c=col, alpha=0.1)[0]
            proj_lines.append(line)
            proj_scatters.append(fish_scatters)

        scatter_projections.append(proj_scatters)
        lines_projections.append(proj_lines)
    # ----------------

    # the text
    all_timestamps = ['f: '+ str(i).zfill(7) for i in range(f0, fE)]
    time_text = ax.text(0, -5, z_h+10,'', fontsize=15)

    # Setting the axes properties
    ax.set_xlim3d([x_l, x_h])
    ax.set_xlabel('X')
    ax.set_ylim3d([y_l, y_h])
    ax.set_ylabel('Y')
    ax.set_zlim3d([z_l, z_h])
    ax.set_zlabel('Z')

    # Provide starting angle for the view
    elevation_angle = 25
    azimuthal_angle = -45
    ax.view_init(elevation_angle, azimuthal_angle)
    
    ani = animation.FuncAnimation(fig, animate_scatters_f1p, numFrames, fargs=(data, scatters, lines, 
                                                                               scatter_projections, lines_projections,
                                                                               time_text, all_timestamps, all_lims),
                                       interval=10, blit=False, repeat=True)

    Writer = animation.writers['ffmpeg']
    writer = Writer(fps=1, metadata=dict(artist='LOS'), bitrate=3500)
    ani.save(savePath, writer=writer)
    return

In [21]:
st_sps[20]

array([120000, 126000])

In [23]:
t0 = time.time()

saveFolder = '/home/liam/temp/fishmovies4/'

if not os.path.exists(saveFolder):
    os.makedirs(saveFolder)
    

st_sps = splitman.start_stop_frames_for_splitdata

for i in range(20,21):
#for i in range(st_sps.shape[0]):
    savePath = os.path.join(saveFolder, splitdata_names[i]+'_mouth_fighting?.mp4')
    print(savePath)

    #f0, fE = st_sps[i]
    f0, fE = st_sps[i,0]+2040, st_sps[i,0]+2106
    n_movFrames = fE - f0
    #movie_tracks = tracks_3D[f0:fE]

    make_f1p_movie(tracks_3D, f0, fE, savePath=savePath)
    print(time.time()-t0)

print('Finished!')
tE = time.time()
print(tE-t0)

/home/liam/temp/fishmovies4/splitdata0029_mouth_fighting?.mp4


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

6.313082218170166
Finished!
6.314618825912476


# Make a 3 panel movie - tank, fishCentred, KDE

## Loading data preamble

In [ ]:
### the frame range of the lin models data from this analysis was 50,000 -> 250,000

In [349]:
## FROM ZEBRAFISH ANALYSIS
sys.path.append('/home/liam/code/Zebrafish-Analysis/lib/')
from loading_data import load_models_data_from_h5
import seaborn as sns
import matplotlib.patches as mpatches
import umap

In [350]:
# define some colors
colors = ['#e6194b', '#3cb44b', '#ffe119', '#4363d8', '#f58231', 
          '#911eb4', '#46f0f0', '#f032e6', '#bcf60c', '#fabebe',
          '#008080', '#e6beff', '#9a6324', '#000000', '#800000',
          '#aaffc3', '#808000', '#ffd8b1', '#000075', '#808080', 
          '#ffffff', '#000000']

In [351]:
## ------ Load the lab coord data --------- ##
##

# load image coordinates data
with h5py.File(tracksPath, 'r') as hf:
    #tracks_3D_raw = hf['tracks_3D'][:]
    tracks_imCoords_raw = hf['tracks_imCoords_raw'][:]
    tracks_3D_smooth = hf['tracks_3D_smooth'][:]
totalnumFrames = tracks_3D_smooth.shape[0]

# define the arrays which hold the edited information
tracks_3D = np.copy(tracks_3D_smooth)

In [352]:
## -------- Load the linear models results --------- ##
##

# load the linear models data
datapath  = '/home/liam/fish_analysis/nugget_dump/20201201/dist_in_pca_basis_entire_zscored.h5'
modelspath = '/home/liam/fish_analysis/nugget_dump/20201201/dist_in_pca_basis_entire_zscored_models.h5'

# loads the timeseries
with h5py.File(datapath, 'r') as hf:
    tseries = hf['tseries'][:]
    
# load the models 
outs = load_models_data_from_h5(modelspath)
numFrames, segment_frame_idxs, symMat, all_models, all_windows = outs
res = [tseries, numFrames, segment_frame_idxs, symMat, all_models, all_windows]
data_ordering = ['tseries', 'numFrames', 'segment_frame_idxs', 'symMat', 'all_models', 'all_windows']

In [511]:
## --------- umap on the dissim mat ----------- #

startTime = time.time()

n_neighbors = 15
min_dist = 0.1
n_components = 2
metric = 'precomputed'
spread=1
title = ''

uMapper = umap.UMAP(learning_rate=0.8, 
                    n_neighbors=n_neighbors, 
                    min_dist=min_dist, 
                    metric='precomputed',
                    n_components=n_components,
                    spread=spread)
standard_embedding = uMapper.fit_transform(symMat)


stopTime = time.time()

print(stopTime - startTime)

/home/liam/anaconda3/envs/FishTank/lib/python3.6/site-packages/umap/umap_.py:1504: UserWarning: using precomputed metric; transform will be unavailable for new data and inverse_transform will be unavailable for all data
  "using precomputed metric; transform will be unavailable for new data and inverse_transform "


9.18351125717163


In [355]:
# ------- Parse down the datas to the same shape ------ ##

startFrame, stopFrame = 50000, 250000
nfs = stopFrame - startFrame

lab_coord_data = np.copy(tracks_3D[startFrame:stopFrame])

tseries_data = np.copy(tseries[startFrame:stopFrame])

pec_coord_data = np.copy(lab_coord_data)
for i in range(lab_coord_data.shape[0]):
    pec_coord_data[i] = transform_frame_positions_to_pec1_basis(lab_coord_data[i])
    
    
# get the timeseries of modelIdxs for each frame
frame_modelIdxs = np.ones((nfs,))*np.NaN
for i,win in enumerate(all_windows):
    frame_modelIdxs[win[0]:win[1]] = i


print(startFrame, stopFrame)
print()

print(lab_coord_data.shape)
print(pec_coord_data.shape)
print(tseries_data.shape)
print(all_windows.shape)
print(standard_embedding.shape)
print(frame_modelIdxs.shape)
print()

50000 250000

(200000, 2, 3, 3)
(200000, 2, 3, 3)
(200000, 9)
(5772, 2)
(5772, 2)
(200000,)



In [ ]:
# plot a KDE of the standard embedding

In [501]:
kwargs_list = [{'levels': np.arange(0.01, 1, 0.05)}]*20
label_patches = []

bw = 0.1

plt.ion()
plt.figure(figsize=(7,5))
plt.title('KDE of LLMs dissimilarity matrix UMAP projection')

sns.kdeplot(x=standard_embedding[:,0],
            y=standard_embedding[:,1],
            color=colors[0],
            alpha=0.5,
            bw_method=bw,
            fill=True,
            **kwargs_list[0]);
label_patch = mpatches.Patch(color=colors[0], label='KDE')
label_patches.append(label_patch)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

## A single frame

In [ ]:
# we need 3 panels -  the 3D lab coordinate aniation, the fish1pec animation, the UMAP animation

In [459]:
#### PLOT A SINGLE FRAME 

fish_colors = ['red', 'blue']

f0 = 0
fE = 10

# -------------------------------------------------------------- #


# set up a figure twice as wide as it is tall
fig = plt.figure(figsize=plt.figaspect(0.3))
#fig = plt.figure()
#plt.tight_layout()


init_f = 0

# ------------ LAB DATA FRAME --------- #

# set up the axes for the first plot
ax1 = fig.add_subplot(1, 3, 1, projection='3d')
ax = ax1
data = np.copy(lab_coord_data[f0:fE])


# Initialize scatters (list over fish, list over bps)
symbols = ['o', 's', 'x']
sizes=12

# set the limits for the plotting.
# This is need to defining the projections too
x_l, x_h = 0, 40
y_l, y_h = 0, 40
z_l, z_h = 0, 40


# ----- Main scatters
scatters = []
for fishIdx in range(numFish):
    col = fish_colors[fishIdx]
    fish_scatters = []
    for bpIdx in range(numBodyPoints):
        fish_scatters.append(ax.scatter(data[init_f,fishIdx,bpIdx,0:1], data[init_f,fishIdx,bpIdx,1:2], data[init_f,fishIdx,bpIdx,2:], 
                                        c=col, s=sizes, marker=symbols[bpIdx])) 
    scatters.append(fish_scatters)

# ------ Main lines
lines = []
for fishIdx in range(numFish):
    col = fish_colors[fishIdx]
    line = ax.plot(data[init_f, fishIdx, :, 0], data[init_f, fishIdx, :, 1], data[init_f, fishIdx, :, 2], c=col)[0]
    lines.append(line)


# ------ projections 
symbols = ['o', 's', 'x']
sizes=5

scatter_projections = []
lines_projections = []
for projectionIdx in range(3):
    
    # THE PATTERN IS NOT THE SAME FOR THE THREE CAMERA
    # IT DEPENDS ON THE VIEW THAT YOU WANT
    
    # XZ => Y=Y+L
    if projectionIdx == 0:
        init_projectionData = np.copy(data[init_f])
        init_projectionData[:, :, 1] = y_l
    # XY => Z=Z_L
    elif projectionIdx == 1:
        init_projectionData = np.copy(data[init_f])
        init_projectionData[:, :, 2] = z_l
    # YZ => X=X_L
    elif projectionIdx == 2:
        init_projectionData = np.copy(data[init_f])
        init_projectionData[:, :, 0] = x_l
    
    proj_scatters = []
    proj_lines = []
    for fishIdx in range(numFish):
        col = fish_colors[fishIdx]
        # scatter plots
        fish_scatters = []
        for bpIdx in range(numBodyPoints):
            fish_scatters.append(ax.scatter(init_projectionData[fishIdx,bpIdx,0:1], 
                                            init_projectionData[fishIdx,bpIdx,1:2], 
                                            init_projectionData[fishIdx,bpIdx,2:], 
                                            c=col, s=sizes, marker=symbols[bpIdx], alpha=0.1))
        # lines
        line = ax.plot(init_projectionData[fishIdx, :, 0], 
                       init_projectionData[fishIdx, :, 1], 
                       init_projectionData[fishIdx, :, 2], c=col, alpha=0.1)[0]
        proj_lines.append(line)
        proj_scatters.append(fish_scatters)

    scatter_projections.append(proj_scatters)
    lines_projections.append(proj_lines)


# -------- the text
all_timestamps = ['frame: '+ str(i).zfill(7) for i in range(f0, fE)]
time_text = ax.text(14, 0, 43,all_timestamps[init_f], fontsize=15)

# Setting the axes properties
ax.set_xlim3d([x_l, x_h])
ax.set_xlabel('X')
ax.set_ylim3d([y_l, y_h])
ax.set_ylabel('Y')
ax.set_zlim3d([z_l, z_h])
ax.set_zlabel('Z')

# Provide starting angle for the view
elevation_angle = 10
azimuthal_angle = 45
ax.view_init(elevation_angle, azimuthal_angle)









# ------------ PEC rep DATA FRAME --------- #

# set up the axes for the first plot
ax2 = fig.add_subplot(1, 3, 2, projection='3d')
ax = ax2
data = np.copy(pec_coord_data[f0:fE])


# Initialize scatters (list over fish, list over bps)
symbols = ['o', 's', 'x']
sizes=12

# set the limits for the plotting.
# This is need to defining the projections too
limval = 10
x_l, x_h = -limval, limval
y_l, y_h = -limval, limval
z_l, z_h = -limval, limval
all_lims = [[x_l, x_h], [y_l, y_h], [z_l, z_h]]


# ----- Main scatters
scatters = []
for fishIdx in range(numFish):
    col = fish_colors[fishIdx]
    fish_scatters = []
    for bpIdx in range(numBodyPoints):
        fish_scatters.append(ax.scatter(data[init_f,fishIdx,bpIdx,0:1], data[init_f,fishIdx,bpIdx,1:2], data[init_f,fishIdx,bpIdx,2:], 
                                        c=col, s=sizes, marker=symbols[bpIdx])) 
    scatters.append(fish_scatters)

# ------ Main lines
lines = []
for fishIdx in range(numFish):
    col = fish_colors[fishIdx]
    line = ax.plot(data[init_f, fishIdx, :, 0], data[init_f, fishIdx, :, 1], data[init_f, fishIdx, :, 2], c=col)[0]
    lines.append(line)


# ------ projections 
symbols = ['o', 's', 'x']
sizes=5

scatter_projections = []
lines_projections = []
for projectionIdx in range(3):
    
    # THE PATTERN IS NOT THE SAME FOR THE THREE CAMERA
    # IT DEPENDS ON THE VIEW THAT YOU WANT
    
    # XZ => Y=Y+L
    if projectionIdx == 0:
        init_projectionData = np.copy(data[init_f])
        init_projectionData[:, :, 1] = -y_l
    # XY => Z=Z_L
    elif projectionIdx == 1:
        init_projectionData = np.copy(data[init_f])
        init_projectionData[:, :, 2] = z_l
    # YZ => X=X_L
    elif projectionIdx == 2:
        init_projectionData = np.copy(data[init_f])
        init_projectionData[:, :, 0] = x_l
    
    proj_scatters = []
    proj_lines = []
    for fishIdx in range(numFish):
        col = fish_colors[fishIdx]
        # scatter plots
        fish_scatters = []
        for bpIdx in range(numBodyPoints):
            fish_scatters.append(ax.scatter(init_projectionData[fishIdx,bpIdx,0:1], 
                                            init_projectionData[fishIdx,bpIdx,1:2], 
                                            init_projectionData[fishIdx,bpIdx,2:], 
                                            c=col, s=sizes, marker=symbols[bpIdx], alpha=0.1))
        # lines
        line = ax.plot(init_projectionData[fishIdx, :, 0], 
                       init_projectionData[fishIdx, :, 1], 
                       init_projectionData[fishIdx, :, 2], c=col, alpha=0.1)[0]
        proj_lines.append(line)
        proj_scatters.append(fish_scatters)

    scatter_projections.append(proj_scatters)
    lines_projections.append(proj_lines)


# Setting the axes properties
ax.set_xlim3d([x_l, x_h])
ax.set_xlabel('X')
ax.set_ylim3d([y_l, y_h])
ax.set_ylabel('Y')
ax.set_zlim3d([z_l, z_h])
ax.set_zlabel('Z')


# Provide starting angle for the view
elevation_angle = 25
azimuthal_angle = -45
ax.view_init(elevation_angle, azimuthal_angle)






# ------------ UMAP --------- #

ax3 = fig.add_subplot(1, 3, 3)
ax = ax3

kwargs_list = [{'levels': np.arange(0.01, 1, 0.05)}]*20
label_patches = []

bw = 0.1

sns.kdeplot(x=standard_embedding[:,0],
            y=standard_embedding[:,1],
            color=colors[0],
            alpha=0.5,
            bw_method=bw,
            fill=True,
            **kwargs_list[0]);
label_patch = mpatches.Patch(color=colors[0], label='KDE')
label_patches.append(label_patch)

# plot the marker of the current frame
modIdx = frame_modelIdxs[init_f]
if ~np.isnan(modIdx):
    ax.scatter(standard_embedding[modIdx,0], standard_embedding[modIdx,1], s=6, color='blue')




Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

## movie

In [348]:
lab_coord_data.shape

NameError: name 'lab_coord_data' is not defined

In [329]:
pec_coord_data.shape

(200000, 2, 3, 3)

In [330]:
standard_embedding.shape

(5772, 2)

In [331]:
frame_modelIdxs.shape

(200000,)

In [370]:
def animate_3panel(iteration, lab_data, lab_scatters, lab_lines, lab_scatter_projections, lab_lines_projections, text, 
                              pec_data, pec_scatters, pec_lines, pec_scatter_projections, pec_lines_projections,
                              frame_mod_data, embed_data, modelScatter, all_timestamps, all_lims_lab, all_lims_pec):
    """
    Update the data held by the scatter plot and therefore animates it.

    Args:
        iteration (int): Current iteration of the animation
        data (list): List of the data positions at each iteration.
        scatters (list): List of all the scatters (One per element)

    Returns:
        list: List of scatters (One per element) with new coordinates
    """
    if np.mod(iteration, 100) == 0:
        print(iteration)
    
    # parse the input shapes
    numFrames, numFish, numBodyPoints, _ = data.shape
 
    #  ---- update the main scatters ------ 
    for fishIdx in range(numFish):
        for bpIdx in range(numBodyPoints):
            lab_scatters[fishIdx][bpIdx]._offsets3d = (lab_data[iteration,fishIdx,bpIdx,0:1],
                                                       lab_data[iteration,fishIdx,bpIdx,1:2],
                                                       lab_data[iteration,fishIdx,bpIdx,2:])
            pec_scatters[fishIdx][bpIdx]._offsets3d = (pec_data[iteration,fishIdx,bpIdx,0:1],
                                                       pec_data[iteration,fishIdx,bpIdx,1:2],
                                                       pec_data[iteration,fishIdx,bpIdx,2:])
            
            
    # ----- update the main lines --------
    for fishIdx,line in enumerate(lab_lines):
        line.set_data(lab_data[iteration, fishIdx, :, :2].swapaxes(0,1))
        line.set_3d_properties(lab_data[iteration, fishIdx, :, 2])
    for fishIdx,line in enumerate(pec_lines):
        line.set_data(pec_data[iteration, fishIdx, :, :2].swapaxes(0,1))
        line.set_3d_properties(pec_data[iteration, fishIdx, :, 2])

        
    # ------- update the text ------------
    #print(iteration)
    text.set_text(all_timestamps[iteration])
    
    
    # ------- update the lab projections ------ #
#     x_l, x_h = all_lims_lab[0][0], all_lims_lab[0][1]
#     y_l, y_h = all_lims_lab[1][0], all_lims_lab[1][1]
#     z_l, z_h = all_lims_lab[2][0], all_lims_lab[2][1]
        
    for projectionIdx in range(3):
        # XZ => Y
        if projectionIdx == 0:
            projectionData = np.copy(lab_data[iteration])
            projectionData[:, :, 1] = 0
        # XY => Z
        elif projectionIdx == 1:
            projectionData = np.copy(lab_data[iteration])
            projectionData[:, :, 2] = 0
        # YZ => X
        elif projectionIdx == 2:
            projectionData = np.copy(lab_data[iteration])
            projectionData[:, :, 0] = 0
            
        proj_scatters = lab_scatter_projections[projectionIdx]
        proj_lines = lab_lines_projections[projectionIdx]
        
        # update the scatters
        for fishIdx in range(numFish):
            for bpIdx in range(numBodyPoints):
                proj_scatters[fishIdx][bpIdx]._offsets3d = (projectionData[fishIdx,bpIdx,0:1],
                                                            projectionData[fishIdx,bpIdx,1:2],
                                                            projectionData[fishIdx,bpIdx,2:])
        # update the lines
        for fishIdx,line in enumerate(proj_lines):
            line.set_data(projectionData[fishIdx, :, :2].swapaxes(0,1))
            line.set_3d_properties(projectionData[fishIdx, :, 2])
            
            
    # ------- update the pec projections ------ #
    x_l, x_h = all_lims_pec[0][0], all_lims_pec[0][1]
    y_l, y_h = all_lims_pec[1][0], all_lims_pec[1][1]
    z_l, z_h = all_lims_pec[2][0], all_lims_pec[2][1]
        
    for projectionIdx in range(3):
        # XZ => Y
        if projectionIdx == 0:
            projectionData = np.copy(pec_data[iteration])
            projectionData[:, :, 1] = y_h
        # XY => Z
        elif projectionIdx == 1:
            projectionData = np.copy(pec_data[iteration])
            projectionData[:, :, 2] = z_l
        # YZ => X
        elif projectionIdx == 2:
            projectionData = np.copy(pec_data[iteration])
            projectionData[:, :, 0] = x_l
            
        proj_scatters = pec_scatter_projections[projectionIdx]
        proj_lines = pec_lines_projections[projectionIdx]
        
        # update the scatters
        for fishIdx in range(numFish):
            for bpIdx in range(numBodyPoints):
                proj_scatters[fishIdx][bpIdx]._offsets3d = (projectionData[fishIdx,bpIdx,0:1],
                                                            projectionData[fishIdx,bpIdx,1:2],
                                                            projectionData[fishIdx,bpIdx,2:])
        # update the lines
        for fishIdx,line in enumerate(proj_lines):
            line.set_data(projectionData[fishIdx, :, :2].swapaxes(0,1))
            line.set_3d_properties(projectionData[fishIdx, :, 2])
            
            
    # ----- update the scatter of the modelIdx ---- #
    
    # plot the marker of the current frame
    modIdx = frame_mod_data[iteration]
    #print(modIdx)
    if np.isnan(modIdx):
        #print('aaa ',iteration, modIdx)
        modelScatter.set_offsets(np.c_[100,100])
    else:
        #print('bbb ', embed_data.shape)
        modelScatter.set_offsets(np.c_[embed_data[int(modIdx),0], embed_data[int(modIdx),1]])
        #print('bbb ')
        #print(np.c_[embed_data[int(modIdx),0], embed_data[int(modIdx),1]])
    return







def make_3panel_movie(lab_coord_data, pec_coord_data, standard_embedding, frame_modelIdxs, f0, fE, savePath='/home/liam/temp/testanimation.mp4'):
    """
    Creates the 3D figure and animates it with the input data.

    Args:
        data (list): List of the data positions at each iteration.
        save (bool): Whether to save the recording of the animation. (Default to False).
    """

    fish_colors = ['red', 'blue']
    numFrames = fE - f0

    # set up a figure twice as wide as it is tall
    fig = plt.figure(figsize=(15,5))


    init_f = 0

    # ------------ LAB DATA FRAME --------- #

    # set up the axes for the first plot
    ax1 = fig.add_subplot(1, 3, 1, projection='3d')
    ax = ax1
    data = np.copy(lab_coord_data[f0:fE])
    
    # Initialize scatters (list over fish, list over bps)
    symbols = ['o', 's', 'x']
    sizes=12

    # set the limits for the plotting.
    # This is need to defining the projections too
    x_l, x_h = 0, 40
    y_l, y_h = 0, 40
    z_l, z_h = 0, 40
    all_lims_lab = [[x_l, x_h], [y_l, y_h], [z_l, z_h]]

    # ----- Main scatters
    lab_scatters = []
    for fishIdx in range(numFish):
        col = fish_colors[fishIdx]
        fish_scatters = []
        for bpIdx in range(numBodyPoints):
            fish_scatters.append(ax.scatter(data[init_f,fishIdx,bpIdx,0:1], data[init_f,fishIdx,bpIdx,1:2], data[init_f,fishIdx,bpIdx,2:], 
                                            c=col, s=sizes, marker=symbols[bpIdx])) 
        lab_scatters.append(fish_scatters)

    # ------ Main lines
    lab_lines = []
    for fishIdx in range(numFish):
        col = fish_colors[fishIdx]
        line = ax.plot(data[init_f, fishIdx, :, 0], data[init_f, fishIdx, :, 1], data[init_f, fishIdx, :, 2], c=col)[0]
        lab_lines.append(line)

    # ------ projections 
    symbols = ['o', 's', 'x']
    sizes=5

    lab_scatter_projections = []
    lab_lines_projections = []
    for projectionIdx in range(3):
        # THE PATTERN IS NOT THE SAME FOR THE THREE CAMERA
        # IT DEPENDS ON THE VIEW THAT YOU WANT
        # XZ => Y=Y+L
        if projectionIdx == 0:
            init_projectionData = np.copy(data[init_f])
            init_projectionData[:, :, 1] = y_l
        # XY => Z=Z_L
        elif projectionIdx == 1:
            init_projectionData = np.copy(data[init_f])
            init_projectionData[:, :, 2] = z_l
        # YZ => X=X_L
        elif projectionIdx == 2:
            init_projectionData = np.copy(data[init_f])
            init_projectionData[:, :, 0] = x_l

        proj_scatters = []
        proj_lines = []
        for fishIdx in range(numFish):
            col = fish_colors[fishIdx]
            # scatter plots
            fish_scatters = []
            for bpIdx in range(numBodyPoints):
                fish_scatters.append(ax.scatter(init_projectionData[fishIdx,bpIdx,0:1], 
                                                init_projectionData[fishIdx,bpIdx,1:2], 
                                                init_projectionData[fishIdx,bpIdx,2:], 
                                                c=col, s=sizes, marker=symbols[bpIdx], alpha=0.1))
            # lines
            line = ax.plot(init_projectionData[fishIdx, :, 0], 
                           init_projectionData[fishIdx, :, 1], 
                           init_projectionData[fishIdx, :, 2], c=col, alpha=0.1)[0]
            proj_lines.append(line)
            proj_scatters.append(fish_scatters)

        lab_scatter_projections.append(proj_scatters)
        lab_lines_projections.append(proj_lines)

    # -------- the text
    all_timestamps = ['frame: '+ str(i).zfill(7) for i in range(f0, fE)]
    time_text = ax.text(14, 0, 43,all_timestamps[init_f], fontsize=15)

    # Setting the axes properties
    ax.set_xlim3d([x_l, x_h])
    ax.set_xlabel('X')
    ax.set_ylim3d([y_l, y_h])
    ax.set_ylabel('Y')
    ax.set_zlim3d([z_l, z_h])
    ax.set_zlabel('Z')

    # Provide starting angle for the view
    elevation_angle = 10
    azimuthal_angle = 45
    ax.view_init(elevation_angle, azimuthal_angle)









    # ------------ PEC rep DATA FRAME --------- #

    # set up the axes for the first plot
    ax2 = fig.add_subplot(1, 3, 2, projection='3d')
    ax = ax2
    data = np.copy(pec_coord_data[f0:fE])

    # Initialize scatters (list over fish, list over bps)
    symbols = ['o', 's', 'x']
    sizes=12

    # set the limits for the plotting.
    # This is need to defining the projections too
    limval = 10
    x_l, x_h = -limval, limval
    y_l, y_h = -limval, limval
    z_l, z_h = -limval, limval
    all_lims_pec = [[x_l, x_h], [y_l, y_h], [z_l, z_h]]

    
    # ----- Main scatters
    pec_scatters = []
    for fishIdx in range(numFish):
        col = fish_colors[fishIdx]
        fish_scatters = []
        for bpIdx in range(numBodyPoints):
            fish_scatters.append(ax.scatter(data[init_f,fishIdx,bpIdx,0:1], data[init_f,fishIdx,bpIdx,1:2], data[init_f,fishIdx,bpIdx,2:], 
                                            c=col, s=sizes, marker=symbols[bpIdx])) 
        pec_scatters.append(fish_scatters)

    # ------ Main lines
    pec_lines = []
    for fishIdx in range(numFish):
        col = fish_colors[fishIdx]
        line = ax.plot(data[init_f, fishIdx, :, 0], data[init_f, fishIdx, :, 1], data[init_f, fishIdx, :, 2], c=col)[0]
        pec_lines.append(line)


    # ------ projections 
    symbols = ['o', 's', 'x']
    sizes=5

    pec_scatter_projections = []
    pec_lines_projections = []
    for projectionIdx in range(3):

        # THE PATTERN IS NOT THE SAME FOR THE THREE CAMERA
        # IT DEPENDS ON THE VIEW THAT YOU WANT

        # XZ => Y=Y+L
        if projectionIdx == 0:
            init_projectionData = np.copy(data[init_f])
            init_projectionData[:, :, 1] = -y_l
        # XY => Z=Z_L
        elif projectionIdx == 1:
            init_projectionData = np.copy(data[init_f])
            init_projectionData[:, :, 2] = z_l
        # YZ => X=X_L
        elif projectionIdx == 2:
            init_projectionData = np.copy(data[init_f])
            init_projectionData[:, :, 0] = x_l

        proj_scatters = []
        proj_lines = []
        for fishIdx in range(numFish):
            col = fish_colors[fishIdx]
            # scatter plots
            fish_scatters = []
            for bpIdx in range(numBodyPoints):
                fish_scatters.append(ax.scatter(init_projectionData[fishIdx,bpIdx,0:1], 
                                                init_projectionData[fishIdx,bpIdx,1:2], 
                                                init_projectionData[fishIdx,bpIdx,2:], 
                                                c=col, s=sizes, marker=symbols[bpIdx], alpha=0.1))
            # lines
            line = ax.plot(init_projectionData[fishIdx, :, 0], 
                           init_projectionData[fishIdx, :, 1], 
                           init_projectionData[fishIdx, :, 2], c=col, alpha=0.1)[0]
            proj_lines.append(line)
            proj_scatters.append(fish_scatters)

        pec_scatter_projections.append(proj_scatters)
        pec_lines_projections.append(proj_lines)


    # Setting the axes properties
    ax.set_xlim3d([x_l, x_h])
    ax.set_xlabel('X')
    ax.set_ylim3d([y_l, y_h])
    ax.set_ylabel('Y')
    ax.set_zlim3d([z_l, z_h])
    ax.set_zlabel('Z')

    # Provide starting angle for the view
    elevation_angle = 25
    azimuthal_angle = -45
    ax.view_init(elevation_angle, azimuthal_angle)






    # ------------ UMAP --------- #

    ax3 = fig.add_subplot(1, 3, 3)
    ax = ax3

    kwargs_list = [{'levels': np.arange(0.01, 1, 0.05)}]*20
    label_patches = []

    bw = 0.1

    sns.kdeplot(x=standard_embedding[:,0],
                y=standard_embedding[:,1],
                color=colors[0],
                alpha=0.5,
                bw_method=bw,
                fill=True,
                **kwargs_list[0]);
    label_patch = mpatches.Patch(color=colors[0], label='KDE')
    label_patches.append(label_patch)

    # plot model of the current frame
    # dummy here - no data (plot off the map)
    modelScatter = ax.scatter(100, 100, s=10, color='blue')
    ax.set_xlim(-1.45, 8.8)
    ax.set_ylim(-0.33, 6.15)
    
    # finally make a paired-down version of embedding data to be passed to updater,
    # to be used to draw the current modelIdx on the UMAP
    #embed_data = np.copy(standard_embedding[f0:fE])
    embed_data = np.copy(standard_embedding)
    frame_mod_data = np.copy(frame_modelIdxs[f0:fE])

    
    
    # -------------------------------------------------------------------------------------------#

    ani = animation.FuncAnimation(fig, animate_3panel, numFrames, fargs=(lab_coord_data[f0:fE], lab_scatters, lab_lines, lab_scatter_projections, lab_lines_projections, time_text, 
                                                                         pec_coord_data[f0:fE], pec_scatters, pec_lines, pec_scatter_projections, pec_lines_projections,
                                                                         frame_mod_data, embed_data, modelScatter, all_timestamps, all_lims_lab, all_lims_pec),
                                       interval=10, blit=False, repeat=True)

    Writer = animation.writers['ffmpeg']
    writer = Writer(fps=100, metadata=dict(artist='LOS'), bitrate=6000)
    ani.save(savePath, writer=writer)
    return

In [ ]:
178000	   the intense interaction is over now. The fish are apart from each other.
	
184900	   one of the fish froze at the bottom of the tank, while the other fish moves around.
	
191600	   one fish "freezes" while the other fish moves around by the frozen fish.
	
192800	   the "frozen" fish moves again.
	
199000	   the fish are facing each other at the bottom of the tank, frozen.

In [364]:
199000 - 178000

21000

In [365]:
178000-50000

128000

In [367]:
199000 - 50000

149000

In [408]:
t0 = time.time()

saveFolder = '/home/liam/temp/fishmovies5/'

if not os.path.exists(saveFolder):
    os.makedirs(saveFolder)
    
f0, fE = 130000, 150000

savePath = os.path.join(saveFolder, '{0}_{1}_beta.mp4'.format(f0, fE))
make_3panel_movie(lab_coord_data, pec_coord_data, standard_embedding, frame_modelIdxs, f0, fE, savePath=savePath)   


print('Finished!')
tE = time.time()
print(tE-t0)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

0
0
100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
1400
1500
1600
1700
1800
1900
2000
2100
2200
2300
2400
2500
2600
2700
2800
2900
3000
3100
3200
3300
3400
3500
3600
3700
3800
3900
4000
4100
4200
4300
4400
4500
4600
4700
4800
4900
5000
5100
5200
5300
5400
5500
5600
5700
5800
5900
6000
6100
6200
6300
6400
6500
6600
6700
6800
6900
7000
7100
7200
7300
7400
7500
7600
7700
7800
7900
8000
8100
8200
8300
8400
8500
8600
8700
8800
8900
9000
9100
9200
9300
9400
9500
9600
9700
9800
9900
10000
10100
10200
10300
10400
10500
10600
10700
10800
10900
11000
11100
11200
11300
11400
11500
11600
11700
11800
11900
12000
12100
12200
12300
12400
12500
12600
12700
12800
12900
13000
13100
13200
13300
13400
13500
13600
13700
13800
13900
14000
14100
14200
14300
14400
14500
14600
14700
14800
14900
15000
15100
15200
15300
15400
15500
15600
15700
15800
15900
16000
16100
16200
16300
16400
16500
16600
16700
16800
16900
17000
17100
17200
17300
17400
17500
17600
17700
17800
17900
18000
18100
18200
18300
18400


In [409]:
t0 = time.time()

saveFolder = '/home/liam/temp/fishmovies5/'

if not os.path.exists(saveFolder):
    os.makedirs(saveFolder)
    
f0, fE = 0, 20000

savePath = os.path.join(saveFolder, '{0}_{1}_beta.mp4'.format(f0, fE))
make_3panel_movie(lab_coord_data, pec_coord_data, standard_embedding, frame_modelIdxs, f0, fE, savePath=savePath)   


print('Finished!')
tE = time.time()
print(tE-t0)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

0
0
100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
1400
1500
1600
1700
1800
1900
2000
2100
2200
2300
2400
2500
2600
2700
2800
2900
3000
3100
3200
3300
3400
3500
3600
3700
3800
3900
4000
4100
4200
4300
4400
4500
4600
4700
4800
4900
5000
5100
5200
5300
5400
5500
5600
5700
5800
5900
6000
6100
6200
6300
6400
6500
6600
6700
6800
6900
7000
7100
7200
7300
7400
7500
7600
7700
7800
7900
8000
8100
8200
8300
8400
8500
8600
8700
8800
8900
9000
9100
9200
9300
9400
9500
9600
9700
9800
9900
10000
10100
10200
10300
10400
10500
10600
10700
10800
10900
11000
11100
11200
11300
11400
11500
11600
11700
11800
11900
12000
12100
12200
12300
12400
12500
12600
12700
12800
12900
13000
13100
13200
13300
13400
13500
13600
13700
13800
13900
14000
14100
14200
14300
14400
14500
14600
14700
14800
14900
15000
15100
15200
15300
15400
15500
15600
15700
15800
15900
16000
16100
16200
16300
16400
16500
16600
16700
16800
16900
17000
17100
17200
17300
17400
17500
17600
17700
17800
17900
18000
18100
18200
18300
18400


## movies from regions of the KDE

In [457]:
standard_embedding

array([[1.3768111 , 4.1176076 ],
       [4.9379544 , 0.83393306],
       [3.5255015 , 0.375557  ],
       ...,
       [5.2413044 , 2.9432254 ],
       [2.7101955 , 4.4739933 ],
       [5.1676145 , 2.961032  ]], dtype=float32)

In [461]:
top_left_region = [-1.2, 4.3, -0.1, 4.9]
mid_right_region = [4.1, 2.5, 6, 3.2]

In [466]:
# create a modelIdxs in both regions

def is_position_in_box(position, box):
    if (position[0] >= box[0]) and (position[0] <= box[2]):
        if (position[1] >= box[1]) and (position[1] <= box[3]):
            return True
    else:
        return False

top_cluster_modIdxs = []
mid_cluster_modIdxs = []

for modIdx in range(standard_embedding.shape[0]):
    pos = standard_embedding[modIdx]
    if is_position_in_box(pos, top_left_region):
        top_cluster_modIdxs.append(modIdx)
    elif is_position_in_box(pos, mid_right_region):
        mid_cluster_modIdxs.append(modIdx)

In [469]:
len(mid_cluster_modIdxs)

230

In [470]:
len(top_cluster_modIdxs)
        

78

In [471]:
# top clusters

t0 = time.time()

saveFolder = '/home/liam/temp/top_regions/'

if not os.path.exists(saveFolder):
    os.makedirs(saveFolder)
    
for modIdx in top_cluster_modIdxs:
    print(modIdx)
    f0, fE = all_windows[modIdx]
    savePath = os.path.join(saveFolder, '{0}.mp4'.format(modIdx))
    make_3panel_movie(lab_coord_data, pec_coord_data, standard_embedding, frame_modelIdxs, f0, fE, savePath=savePath)   


print('Finished!')
tE = time.time()
print(tE-t0)

11


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

0
0
100
217


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

0
0
240


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

... (147 outputs left out)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

0
0
5758


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

0
0
Finished!
1657.1610860824585


In [474]:
# top clusters

t0 = time.time()

saveFolder = '/home/liam/temp/mid_regions/'

if not os.path.exists(saveFolder):
    os.makedirs(saveFolder)
    
for modIdx in mid_cluster_modIdxs[:4]:
    print(modIdx)
    f0, fE = all_windows[modIdx]
    savePath = os.path.join(saveFolder, '{0}.mp4'.format(modIdx))
    make_3panel_movie(lab_coord_data, pec_coord_data, standard_embedding, frame_modelIdxs, f0, fE, savePath=savePath)   


print('Finished!')
tE = time.time()
print(tE-t0)

9


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

0
0
10


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

0
0
39


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

0
0
169


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

0
0
Finished!
50.492549657821655


# Properly segmenting the UMAP

In [740]:
## -------- Load the linear models results --------- ##
##

# load the linear models data
datapath  = '/home/liam/fish_analysis/nugget_dump/20201201/dist_in_pca_basis_entire_zscored.h5'
modelspath = '/home/liam/fish_analysis/nugget_dump/20201201/dist_in_pca_basis_entire_zscored_models.h5'

# loads the timeseries
with h5py.File(datapath, 'r') as hf:
    tseries = hf['tseries'][:]
    
# load the models 
outs = load_models_data_from_h5(modelspath)
numFrames, segment_frame_idxs, symMat, all_models, all_windows = outs
res = [tseries, numFrames, segment_frame_idxs, symMat, all_models, all_windows]
data_ordering = ['tseries', 'numFrames', 'segment_frame_idxs', 'symMat', 'all_models', 'all_windows']

In [741]:
## --------- umap on the dissim mat ----------- #

startTime = time.time()

n_neighbors = 15
min_dist = 0.1
n_components = 2
metric = 'precomputed'
spread=1
title = ''

uMapper = umap.UMAP(learning_rate=0.8, 
                    n_neighbors=n_neighbors, 
                    min_dist=min_dist, 
                    metric='precomputed',
                    n_components=n_components,
                    spread=spread)
standard_embedding = uMapper.fit_transform(symMat)


stopTime = time.time()

print(stopTime - startTime)

/home/liam/anaconda3/envs/FishTank/lib/python3.6/site-packages/umap/umap_.py:1504: UserWarning: using precomputed metric; transform will be unavailable for new data and inverse_transform will be unavailable for all data
  "using precomputed metric; transform will be unavailable for new data and inverse_transform "


15.557939767837524


## Perform the KDE

In [742]:
x = np.copy(standard_embedding[:,0])
y = np.copy(standard_embedding[:,1])
xmin, xmax = np.min(x), np.max(x)
ymin, ymax = np.min(y), np.max(y)

# Peform the kernel density estimate
xx, yy = np.mgrid[xmin:xmax:100j, ymin:ymax:100j]
positions = np.vstack([xx.ravel(), yy.ravel()])
values = np.vstack([x, y])
kernel = st.gaussian_kde(values, bw_method=0.1)
f = np.reshape(kernel(positions).T, xx.shape)

### Plot a KDE using scipy and numpy: imshow


In [ ]:
# https://stackoverflow.com/a/30146280

In [115]:
np.min(f)

2.2273978148004284e-82

In [743]:
x = np.copy(standard_embedding[:,0])
y = np.copy(standard_embedding[:,1])
xmin, xmax = np.min(x), np.max(x)
ymin, ymax = np.min(y), np.max(y)

# Peform the kernel density estimate
xx, yy = np.mgrid[xmin:xmax:100j, ymin:ymax:100j]
positions = np.vstack([xx.ravel(), yy.ravel()])
values = np.vstack([x, y])
kernel = st.gaussian_kde(values, bw_method=0.1)
f = np.reshape(kernel(positions).T, xx.shape)

fig = plt.figure(figsize=(8,6))
ax = fig.gca()
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
# kernel density estimate plot instead of the contourf plot
ax.imshow(np.rot90(f), cmap='Blues', extent=[xmin, xmax, ymin, ymax])



Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [744]:
plt.figure()
plt.pcolormesh(xx, yy, f, cmap='jet')
plt.colorbar()
#plt.scatter(x, y, s=2, facecolor='white')

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

/home/liam/anaconda3/envs/FishTank/lib/python3.6/site-packages/ipykernel_launcher.py:2: MatplotlibDeprecationWarning: shading='flat' when X and Y have the same dimensions as C is deprecated since 3.3.  Either specify the corners of the quadrilaterals with X and Y, or pass shading='auto', 'nearest' or 'gouraud', or set rcParams['pcolor.shading'].  This will become an error two minor releases later.
  


### Plot a KDE using scipy and numpy: contourf

In [745]:
x = np.copy(standard_embedding[:,0])
y = np.copy(standard_embedding[:,1])
xmin, xmax = np.min(x), np.max(x)
ymin, ymax = np.min(y), np.max(y)

# Peform the kernel density estimate
xx, yy = np.mgrid[xmin:xmax:100j, ymin:ymax:100j]
positions = np.vstack([xx.ravel(), yy.ravel()])
values = np.vstack([x, y])
kernel = st.gaussian_kde(values, bw_method=0.1)
f = np.reshape(kernel(positions).T, xx.shape)

fig = pl.figure()
ax = fig.gca()
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
# Contourf plot
cfset = ax.contourf(xx, yy, f, levels=30, cmap='Blues')
# Contour plot
cset = ax.contour(xx, yy, f, colors='k')
# Label plot
ax.clabel(cset, inline=1, fontsize=10)
#ax.set_xlabel('Y1')
#ax.set_ylabel('Y0')

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

<a list of 16 text.Text objects>

### Plot a KDE using scipy and numpy: 3D

In [746]:
fig = plt.figure(figsize=(8, 5))
ax = plt.axes(projection='3d')
surf = ax.plot_surface(xx, yy, f, rstride=1, cstride=1, cmap='coolwarm', edgecolor='none')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('PDF')
ax.set_title('Surface plot of Gaussian 2D KDE')
fig.colorbar(surf, shrink=0.5, aspect=5) # add color bar indicating the PDF
ax.view_init(60, 35)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

## Thresholding

In [747]:
# threshold

# anything less than max_val*thresh_fac goes to 0
thresh_fac = 0.7

thresh = np.max(f)*thresh_fac

zz_threshed = np.copy(f)

# # first plot with just the zeros
zz_threshed[zz_threshed < thresh] = 0
plt.figure()
plt.pcolormesh(xx, yy, zz_threshed, shading='flat', cmap='jet')
# plt.colorbar()

# now fully threshold (others to 1)
zz_threshed[zz_threshed != 0] = 1
plt.figure()
plt.pcolormesh(xx, yy, zz_threshed, cmap='jet')

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

/home/liam/anaconda3/envs/FishTank/lib/python3.6/site-packages/ipykernel_launcher.py:13: MatplotlibDeprecationWarning: shading='flat' when X and Y have the same dimensions as C is deprecated since 3.3.  Either specify the corners of the quadrilaterals with X and Y, or pass shading='auto', 'nearest' or 'gouraud', or set rcParams['pcolor.shading'].  This will become an error two minor releases later.
  del sys.path[0]


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

/home/liam/anaconda3/envs/FishTank/lib/python3.6/site-packages/ipykernel_launcher.py:19: MatplotlibDeprecationWarning: shading='flat' when X and Y have the same dimensions as C is deprecated since 3.3.  Either specify the corners of the quadrilaterals with X and Y, or pass shading='auto', 'nearest' or 'gouraud', or set rcParams['pcolor.shading'].  This will become an error two minor releases later.


In [748]:
##  ---- find connected components ----- 
#
# define a structure for finding connections (we will use general form)
# then find the binIdxs corresponding to each cluster

from scipy.ndimage.measurements import label

# make the structure
structure = np.ones((3, 3), dtype=np.int) 

# find the connected components
labeled, ncomponents = label(zz_threshed, structure)
#print(ncomponents)

# # create the indices for each clusters, subject to a threshold on size
size_thresh = 5
indices = np.indices(zz_threshed.shape).T[:,:,[1, 0]]
numClusters = 0
cluster_bin_idxs = []
cluster_grid_locs = []
cluster_idxs = []
for compIdx in range(1, ncomponents+1):
    numBins_for_cluster = indices[labeled==compIdx].shape[0]
    if numBins_for_cluster >= size_thresh:
        cluster_idxs.append(numClusters)
        cluster_bin_idxs.append(indices[labeled==compIdx])
        numClusters += 1
        # loop over the last entry, converting binIdxs to locations in UMAP coords
        bins_blcs = []
        for binIdx in cluster_bin_idxs[-1]:
            bin_blc = np.array([xx[binIdx[0], 0], yy[0, binIdx[1]]])
            bins_blcs.append(bin_blc)
        cluster_grid_locs.append(np.vstack(bins_blcs))


# find the spacings between grid points
dx = xx[1, 0] - xx[0, 0]
dy = yy[0, 1] - yy[0, 0]

In [755]:
### Get cluster_labels


# prepare an array which will be analagous to cluster_labels
model_cluster_idxs = np.ones((all_models.shape[0],), dtype=int)*np.NaN

# try and assign an identity to each model
for modelIdx in range(standard_embedding.shape[0]):
    
    model_grid_location = standard_embedding[modelIdx, :]
    
    # compare model location to each bin of each cluster
    for clusterIdx in range(len(cluster_grid_locs)):
        cluster_locs = cluster_grid_locs[clusterIdx]
        for grid_loc_idx in range(cluster_locs.shape[0]):
            grid_loc = cluster_locs[grid_loc_idx]
            
            # is this model in this bin? Then assign the clusterIdx+1 to this model
            if (model_grid_location[0] >= grid_loc[0]) and (model_grid_location[0] <= grid_loc[0]+dx):
                if (model_grid_location[1] >= grid_loc[1]) and (model_grid_location[1] <= grid_loc[1]+dy):
                    model_cluster_idxs[modelIdx] = clusterIdx + 1 # move to 1-indexing from 0-indexing
                    continue
            
    

In [ ]:
# top movies from the first cluster

t0 = time.time()

cluster1_mod_idxs = np.where(model_cluster_idxs==1)[0]

saveFolder = '/home/liam/temp/cluster1/'

if not os.path.exists(saveFolder):
    os.makedirs(saveFolder)
    
for modIdx in cluster1_mod_idxs:
    print(modIdx)
    f0, fE = all_windows[modIdx]
    savePath = os.path.join(saveFolder, '{0}.mp4'.format(modIdx))
    make_3panel_movie(lab_coord_data, pec_coord_data, standard_embedding, frame_modelIdxs, f0, fE, savePath=savePath)   


print('Finished!')
tE = time.time()
print(tE-t0)

In [362]:
np.nanmax(model_cluster_idxs)

5.0

In [363]:
np.nanmin(model_cluster_idxs)

1.0

In [751]:
cluster1_mod_idxs = np.where(model_cluster_idxs==1)[0]
cluster1_mod_idxs

array([  20,  101,  111,  112,  113,  130,  135,  163,  164,  179,  180,
        181,  223,  224,  225,  236,  250,  256,  259,  325,  326,  370,
        384,  385,  413,  425,  435,  443,  460,  470,  471,  474,  488,
        492,  497,  503,  504,  512,  539,  541,  542,  565,  592,  631,
        645,  651,  672,  673,  681,  693,  697,  698,  714,  728,  741,
        762,  789,  790,  795,  822,  823,  849,  870,  878,  885,  886,
        887,  888,  900,  903,  904,  920,  921,  971,  981,  993, 1010,
       1020, 1024, 1025, 1031, 1032, 1042, 1070, 1082, 1109, 1114, 1122,
       1136, 1149, 1150, 1155, 1157, 1163, 1169, 1186, 1187, 1231, 1271,
       1320, 1344, 1363, 1375, 1385, 1391, 1392, 1393, 1402, 1403, 1415,
       1431, 1432, 1450, 1452, 1481, 1482, 1483, 1500, 1515, 1550, 1583,
       1595, 1596, 1620, 1629, 1630, 1631, 1640, 1661, 1665, 1678, 1683,
       1684, 1691, 1693, 1694, 1711, 1713, 1724, 1728, 1745, 1752, 1756,
       1767, 1768, 1770, 1771, 1803, 1819, 1821, 18

# Make the 5-cluster movie

The idea is make 5 separate movies, and a nice still frame of the UMAP showing the clusters.

Then we will have a second step where we combine the 5 movies and image into a single movie. 

## Step 0: prepare the data

In [366]:
## -------- Load the symmat --------- ##

# load the linear models data
datapath  = '/home/liam/fish_analysis/nugget_dump/20201201/dist_in_pca_basis_entire_zscored.h5'
modelspath = '/home/liam/fish_analysis/nugget_dump/20201201/dist_in_pca_basis_entire_zscored_models.h5'
    
# load the models 
outs = load_models_data_from_h5(modelspath)
numFrames, segment_frame_idxs, symMat, all_models, all_windows = outs
res = [tseries, numFrames, segment_frame_idxs, symMat, all_models, all_windows]
data_ordering = ['tseries', 'numFrames', 'segment_frame_idxs', 'symMat', 'all_models', 'all_windows']

symMat.shape


(5772, 5772)

In [367]:
## --------- umap on the dissim mat ----------- #

startTime = time.time()

n_neighbors = 15
min_dist = 0.1
n_components = 2
metric = 'precomputed'
spread=1
title = ''

uMapper = umap.UMAP(learning_rate=0.8, 
                    n_neighbors=n_neighbors, 
                    min_dist=min_dist, 
                    metric='precomputed',
                    n_components=n_components,
                    spread=spread)
standard_embedding = uMapper.fit_transform(symMat)


stopTime = time.time()

print(stopTime - startTime)

/home/liam/anaconda3/envs/FishTank/lib/python3.6/site-packages/umap/umap_.py:1504: UserWarning: using precomputed metric; transform will be unavailable for new data and inverse_transform will be unavailable for all data
  "using precomputed metric; transform will be unavailable for new data and inverse_transform "


10.891162633895874


In [365]:
# ------- Parse down the datas to the same shape ------ ##

startFrame, stopFrame = 50000, 250000
nfs = stopFrame - startFrame

lab_coord_data = np.copy(tracks_3D[startFrame:stopFrame])

tseries_data = np.copy(tseries[startFrame:stopFrame])

pec_coord_data = np.copy(lab_coord_data)
for i in range(lab_coord_data.shape[0]):
    pec_coord_data[i] = transform_frame_positions_to_pec1_basis(lab_coord_data[i])
    
    
# get the timeseries of modelIdxs for each frame
frame_modelIdxs = np.ones((nfs,))*np.NaN
for i,win in enumerate(all_windows):
    frame_modelIdxs[win[0]:win[1]] = i


print(startFrame, stopFrame)
print()

print(lab_coord_data.shape)
print(pec_coord_data.shape)
print(tseries_data.shape)
print(all_windows.shape)
print(standard_embedding.shape)
print(frame_modelIdxs.shape)
print()

50000 250000

(200000, 2, 3, 3)
(200000, 2, 3, 3)
(200000, 9)
(5772, 2)
(5772, 2)
(200000,)



## Step 1: write the 5 movies

In [364]:
# TODO: use a variant of 10.4

In [426]:
def animate_2panel(iteration, lab_data, lab_scatters, lab_lines, lab_scatter_projections, lab_lines_projections, text, 
                              pec_data, pec_scatters, pec_lines, pec_scatter_projections, pec_lines_projections,
                              all_timestamps, all_lims_lab, all_lims_pec):
    """
    Update the data held by the scatter plot and therefore animates it.

    Args:
        iteration (int): Current iteration of the animation
        data (list): List of the data positions at each iteration.
        scatters (list): List of all the scatters (One per element)

    Returns:
        list: List of scatters (One per element) with new coordinates
    """
    #if np.mod(iteration, 100) == 0:
        #print(iteration)
    
    # parse the input shapes
    numFrames, numFish, numBodyPoints, _ = lab_data.shape
 
    #  ---- update the main scatters ------ 
    for fishIdx in range(numFish):
        for bpIdx in range(numBodyPoints):
            lab_scatters[fishIdx][bpIdx]._offsets3d = (lab_data[iteration,fishIdx,bpIdx,0:1],
                                                       lab_data[iteration,fishIdx,bpIdx,1:2],
                                                       lab_data[iteration,fishIdx,bpIdx,2:])
            pec_scatters[fishIdx][bpIdx]._offsets3d = (pec_data[iteration,fishIdx,bpIdx,0:1],
                                                       pec_data[iteration,fishIdx,bpIdx,1:2],
                                                       pec_data[iteration,fishIdx,bpIdx,2:])
            
            
    # ----- update the main lines --------
    for fishIdx,line in enumerate(lab_lines):
        line.set_data(lab_data[iteration, fishIdx, :, :2].swapaxes(0,1))
        line.set_3d_properties(lab_data[iteration, fishIdx, :, 2])
    for fishIdx,line in enumerate(pec_lines):
        line.set_data(pec_data[iteration, fishIdx, :, :2].swapaxes(0,1))
        line.set_3d_properties(pec_data[iteration, fishIdx, :, 2])

        
    # ------- update the text ------------
    #print(iteration)
    text.set_text(all_timestamps[iteration])
    
    
    # ------- update the lab projections ------ #
    for projectionIdx in range(3):
        # XZ => Y
        if projectionIdx == 0:
            projectionData = np.copy(lab_data[iteration])
            projectionData[:, :, 1] = 0
        # XY => Z
        elif projectionIdx == 1:
            projectionData = np.copy(lab_data[iteration])
            projectionData[:, :, 2] = 0
        # YZ => X
        elif projectionIdx == 2:
            projectionData = np.copy(lab_data[iteration])
            projectionData[:, :, 0] = 0
            
        proj_scatters = lab_scatter_projections[projectionIdx]
        proj_lines = lab_lines_projections[projectionIdx]
        
        # update the scatters
        for fishIdx in range(numFish):
            for bpIdx in range(numBodyPoints):
                proj_scatters[fishIdx][bpIdx]._offsets3d = (projectionData[fishIdx,bpIdx,0:1],
                                                            projectionData[fishIdx,bpIdx,1:2],
                                                            projectionData[fishIdx,bpIdx,2:])
        # update the lines
        for fishIdx,line in enumerate(proj_lines):
            line.set_data(projectionData[fishIdx, :, :2].swapaxes(0,1))
            line.set_3d_properties(projectionData[fishIdx, :, 2])
            
            
    # ------- update the pec projections ------ #
    x_l, x_h = all_lims_pec[0][0], all_lims_pec[0][1]
    y_l, y_h = all_lims_pec[1][0], all_lims_pec[1][1]
    z_l, z_h = all_lims_pec[2][0], all_lims_pec[2][1]
        
    for projectionIdx in range(3):
        # XZ => Y
        if projectionIdx == 0:
            projectionData = np.copy(pec_data[iteration])
            projectionData[:, :, 1] = y_h
        # XY => Z
        elif projectionIdx == 1:
            projectionData = np.copy(pec_data[iteration])
            projectionData[:, :, 2] = z_l
        # YZ => X
        elif projectionIdx == 2:
            projectionData = np.copy(pec_data[iteration])
            projectionData[:, :, 0] = x_l
            
        proj_scatters = pec_scatter_projections[projectionIdx]
        proj_lines = pec_lines_projections[projectionIdx]
        
        # update the scatters
        for fishIdx in range(numFish):
            for bpIdx in range(numBodyPoints):
                proj_scatters[fishIdx][bpIdx]._offsets3d = (projectionData[fishIdx,bpIdx,0:1],
                                                            projectionData[fishIdx,bpIdx,1:2],
                                                            projectionData[fishIdx,bpIdx,2:])
        # update the lines
        for fishIdx,line in enumerate(proj_lines):
            line.set_data(projectionData[fishIdx, :, :2].swapaxes(0,1))
            line.set_3d_properties(projectionData[fishIdx, :, 2])
            
    return







def make_2panel_movie(lab_coord_data, pec_coord_data, f0, fE, clusterIdx, savePath='/home/liam/temp/testanimation.mp4'):
    """
    Creates the 3D figure and animates it with the input data.

    Args:
        data (list): List of the data positions at each iteration.
        save (bool): Whether to save the recording of the animation. (Default to False).
    """
    plt.ioff()

    fish_colors = ['red', 'blue']
    numFrames = fE - f0

    # set up a figure twice as wide as it is tall
    #fig = plt.figure(dpi=900)
    #w, h = matplotlib.figure.figaspect(2.)
    fig = plt.figure(figsize=(10, 4))
    fig.tight_layout()
    fig.suptitle('Cluster {0}'.format(clusterIdx), fontsize=16)
    
    #TODO
    # remove this variable, we only use it as a placeholder 
    # for creating figures before passing to the animators
    init_f = 0

    # ------------ LAB DATA FRAME --------- #
    

    # set up the axes for the first plot
    ax1 = fig.add_subplot(1, 2, 1, projection='3d')
    ax = ax1
    data = np.copy(lab_coord_data[f0:fE])
    
    # Initialize scatters (list over fish, list over bps)
    symbols = ['o', 's', 'x']
    sizes=12

    # set the limits for the plotting.
    # This is need to defining the projections too
    x_l, x_h = 0, 40
    y_l, y_h = 0, 40
    z_l, z_h = 0, 40
    all_lims_lab = [[x_l, x_h], [y_l, y_h], [z_l, z_h]]

    # ----- Main scatters
    lab_scatters = []
    for fishIdx in range(numFish):
        col = fish_colors[fishIdx]
        fish_scatters = []
        for bpIdx in range(numBodyPoints):
            fish_scatters.append(ax.scatter(data[init_f,fishIdx,bpIdx,0:1], data[init_f,fishIdx,bpIdx,1:2], data[init_f,fishIdx,bpIdx,2:], 
                                            c=col, s=sizes, marker=symbols[bpIdx])) 
        lab_scatters.append(fish_scatters)

    # ------ Main lines
    lab_lines = []
    for fishIdx in range(numFish):
        col = fish_colors[fishIdx]
        line = ax.plot(data[init_f, fishIdx, :, 0], data[init_f, fishIdx, :, 1], data[init_f, fishIdx, :, 2], c=col)[0]
        lab_lines.append(line)

    # ------ projections 
    symbols = ['o', 's', 'x']
    sizes=5

    lab_scatter_projections = []
    lab_lines_projections = []
    for projectionIdx in range(3):
        # THE PATTERN IS NOT THE SAME FOR THE THREE CAMERA
        # IT DEPENDS ON THE VIEW THAT YOU WANT
        # XZ => Y=Y+L
        if projectionIdx == 0:
            init_projectionData = np.copy(data[init_f])
            init_projectionData[:, :, 1] = y_l
        # XY => Z=Z_L
        elif projectionIdx == 1:
            init_projectionData = np.copy(data[init_f])
            init_projectionData[:, :, 2] = z_l
        # YZ => X=X_L
        elif projectionIdx == 2:
            init_projectionData = np.copy(data[init_f])
            init_projectionData[:, :, 0] = x_l

        proj_scatters = []
        proj_lines = []
        for fishIdx in range(numFish):
            col = fish_colors[fishIdx]
            # scatter plots
            fish_scatters = []
            for bpIdx in range(numBodyPoints):
                fish_scatters.append(ax.scatter(init_projectionData[fishIdx,bpIdx,0:1], 
                                                init_projectionData[fishIdx,bpIdx,1:2], 
                                                init_projectionData[fishIdx,bpIdx,2:], 
                                                c=col, s=sizes, marker=symbols[bpIdx], alpha=0.1))
            # lines
            line = ax.plot(init_projectionData[fishIdx, :, 0], 
                           init_projectionData[fishIdx, :, 1], 
                           init_projectionData[fishIdx, :, 2], c=col, alpha=0.1)[0]
            proj_lines.append(line)
            proj_scatters.append(fish_scatters)

        lab_scatter_projections.append(proj_scatters)
        lab_lines_projections.append(proj_lines)

    # -------- the text
    all_timestamps = ['frame: '+ str(i).zfill(7) for i in range(f0, fE)]
    time_text = ax.text(14, 0, 43,all_timestamps[init_f], fontsize=15)

    # Setting the axes properties
    ax.set_xlim3d([x_l, x_h])
    ax.set_xlabel('X')
    ax.set_ylim3d([y_l, y_h])
    ax.set_ylabel('Y')
    ax.set_zlim3d([z_l, z_h])
    ax.set_zlabel('Z')

    # Provide starting angle for the view
    elevation_angle = 10
    azimuthal_angle = 45
    ax.view_init(elevation_angle, azimuthal_angle)









    # ------------ PEC rep DATA FRAME --------- #

    # set up the axes for the first plot
    ax2 = fig.add_subplot(1, 2, 2, projection='3d')
    ax = ax2
    data = np.copy(pec_coord_data[f0:fE])

    # Initialize scatters (list over fish, list over bps)
    symbols = ['o', 's', 'x']
    sizes=12

    # set the limits for the plotting.
    # This is need to defining the projections too
    limval = 10
    x_l, x_h = -limval, limval
    y_l, y_h = -limval, limval
    z_l, z_h = -limval, limval
    all_lims_pec = [[x_l, x_h], [y_l, y_h], [z_l, z_h]]

    
    # ----- Main scatters
    pec_scatters = []
    for fishIdx in range(numFish):
        col = fish_colors[fishIdx]
        fish_scatters = []
        for bpIdx in range(numBodyPoints):
            fish_scatters.append(ax.scatter(data[init_f,fishIdx,bpIdx,0:1], data[init_f,fishIdx,bpIdx,1:2], data[init_f,fishIdx,bpIdx,2:], 
                                            c=col, s=sizes, marker=symbols[bpIdx])) 
        pec_scatters.append(fish_scatters)

    # ------ Main lines
    pec_lines = []
    for fishIdx in range(numFish):
        col = fish_colors[fishIdx]
        line = ax.plot(data[init_f, fishIdx, :, 0], data[init_f, fishIdx, :, 1], data[init_f, fishIdx, :, 2], c=col)[0]
        pec_lines.append(line)


    # ------ projections 
    symbols = ['o', 's', 'x']
    sizes=5

    pec_scatter_projections = []
    pec_lines_projections = []
    for projectionIdx in range(3):

        # THE PATTERN IS NOT THE SAME FOR THE THREE CAMERA
        # IT DEPENDS ON THE VIEW THAT YOU WANT

        # XZ => Y=Y+L
        if projectionIdx == 0:
            init_projectionData = np.copy(data[init_f])
            init_projectionData[:, :, 1] = -y_l
        # XY => Z=Z_L
        elif projectionIdx == 1:
            init_projectionData = np.copy(data[init_f])
            init_projectionData[:, :, 2] = z_l
        # YZ => X=X_L
        elif projectionIdx == 2:
            init_projectionData = np.copy(data[init_f])
            init_projectionData[:, :, 0] = x_l

        proj_scatters = []
        proj_lines = []
        for fishIdx in range(numFish):
            col = fish_colors[fishIdx]
            # scatter plots
            fish_scatters = []
            for bpIdx in range(numBodyPoints):
                fish_scatters.append(ax.scatter(init_projectionData[fishIdx,bpIdx,0:1], 
                                                init_projectionData[fishIdx,bpIdx,1:2], 
                                                init_projectionData[fishIdx,bpIdx,2:], 
                                                c=col, s=sizes, marker=symbols[bpIdx], alpha=0.1))
            # lines
            line = ax.plot(init_projectionData[fishIdx, :, 0], 
                           init_projectionData[fishIdx, :, 1], 
                           init_projectionData[fishIdx, :, 2], c=col, alpha=0.1)[0]
            proj_lines.append(line)
            proj_scatters.append(fish_scatters)

        pec_scatter_projections.append(proj_scatters)
        pec_lines_projections.append(proj_lines)


    # Setting the axes properties
    ax.set_xlim3d([x_l, x_h])
    #ax.set_xlabel('X')
    ax.set_ylim3d([y_l, y_h])
    #ax.set_ylabel('Y')
    ax.set_zlim3d([z_l, z_h])
    #ax.set_zlabel('Z')
    #ax.set_xticks([])
    #ax.set_xticks([], minor=True)
    #ax.set_yticks([], minor=True)
    #ax.set_zticks([], minor=True)
    ax.set_xticks([-2, -4, 0, 2, 4])
    ax.set_yticks([-2, -4, 0, 2, 4])
    ax.set_zticks([-2, -4, 0, 2, 4])

    # Provide starting angle for the view
    elevation_angle = 25
    azimuthal_angle = -45
    ax.view_init(elevation_angle, azimuthal_angle)



    
    
    # -------------------------------------------------------------------------------------------#
    
    fig.subplots_adjust(left=0.1, bottom=0, right=0.9, top=1, wspace=None, hspace=None)

    ani = animation.FuncAnimation(fig, animate_2panel, numFrames, fargs=(lab_coord_data[f0:fE], lab_scatters, lab_lines, lab_scatter_projections, lab_lines_projections, time_text, 
                                                                         pec_coord_data[f0:fE], pec_scatters, pec_lines, pec_scatter_projections, pec_lines_projections,
                                                                         all_timestamps, all_lims_lab, all_lims_pec),
                                       interval=10, blit=False, repeat=True)

    Writer = animation.writers['ffmpeg']
    writer = Writer(fps=100, metadata=dict(artist='LOS'), bitrate=6000)
    ani.save(savePath, writer=writer)
    plt.ion()
    return

In [428]:
# Make cluster 1 movie

t0 = time.time()
#plt.ioff()

clusterIdx = 0
saveFolder = '/home/liam/temp/clusters/1/'
cluster_mod_idxs = np.where(model_cluster_idxs==1)[0]

# -------- # 
if not os.path.exists(saveFolder):
    os.makedirs(saveFolder)
    
for i,modIdx in enumerate(cluster_mod_idxs):
    print(i, modIdx)
    f0, fE = all_windows[modIdx]
    savePath = os.path.join(saveFolder, '{0}.mp4'.format(modIdx))
    make_2panel_movie(lab_coord_data, pec_coord_data, f0, fE, clusterIdx, savePath=savePath)   


print('Finished!')
tE = time.time()
#plt.ion()
print(tE-t0)

0 20
1 111
2 112
3 118
4 124
5 130
6 135
7 163
8 164
9 180
10 181
11 223
12 224
13 225
14 236
15 250
16 256
17 259
18 351
19 370
20 413
21 425
22 426
23 435
24 443
25 460
26 471
27 473
28 474
29 488
30 491
31 492
32 497
33 503
34 512
35 539
36 541
37 592
38 613
39 631
40 645
41 693
42 697
43 698
44 714
45 728
46 741
47 762
48 784
49 789
50 790
51 795
52 796
53 823
54 849
55 878
56 885
57 886
58 887
59 888
60 900
61 903
62 920
63 921
64 971
65 972
66 981
67 993
68 1010
69 1020
70 1024
71 1026
72 1031
73 1032
74 1070
75 1082
76 1109
77 1136
78 1137
79 1150
80 1155
81 1157
82 1163
83 1169
84 1186
85 1187
86 1231
87 1314
88 1319
89 1320
90 1344
91 1363
92 1375
93 1385
94 1391
95 1392
96 1393
97 1402
98 1403
99 1415
100 1417
101 1432
102 1450
103 1452
104 1478
105 1481
106 1482
107 1483
108 1515
109 1550
110 1552
111 1583
112 1595
113 1596
114 1620
115 1629
116 1630
117 1631
118 1640
119 1665
120 1669
121 1678
122 1683
123 1684
124 1691
125 1701
126 1711
127 1713
128 1724
129 1745
130 1751


In [431]:
numClusters

5

In [432]:
# Make cluster 1 movie

t0 = time.time()
#plt.ioff()

for clusterIdx in range(numClusters):
    print('clusterIdx {0}'.format(clusterIdx))
    print()
    
    # temp test since we did first cluster already
    if clusterIdx == 0:
        continue

    saveFolder = '/home/liam/temp/clusters/' + str(clusterIdx+1) + '/'
    cluster_mod_idxs = np.where(model_cluster_idxs==clusterIdx+1)[0]

    # -------- # 
    if not os.path.exists(saveFolder):
        os.makedirs(saveFolder)

    for i,modIdx in enumerate(cluster_mod_idxs):
        print(i, modIdx)
        f0, fE = all_windows[modIdx]
        savePath = os.path.join(saveFolder, '{0}.mp4'.format(modIdx))
        make_2panel_movie(lab_coord_data, pec_coord_data, f0, fE, clusterIdx, savePath=savePath)  


print('Finished!')
tE = time.time()
#plt.ion()
print(tE-t0)

clusterIdx 0

clusterIdx 1

0 3
1 8
2 15
3 23
4 25
5 33
6 44
7 66
8 88
9 94
10 95
11 99
12 104
13 105
14 115
15 121
16 132
17 133
18 144
19 173
20 174
21 185
22 196
23 202
24 209
25 210
26 229
27 231
28 235
29 246
30 253
31 255
32 261
33 269
34 271
35 279
36 283
37 324
38 328
39 329
40 346
41 350
42 356
43 357
44 359
45 364
46 365
47 366
48 373
49 381
50 382
51 390
52 393
53 404
54 405
55 420
56 429
57 431
58 445
59 446
60 451
61 459
62 467
63 478
64 482
65 495
66 496
67 500
68 507
69 516
70 517
71 518
72 536
73 543
74 545
75 568
76 573
77 575
78 576
79 580
80 602
81 618
82 628
83 632
84 636
85 658
86 686
87 702
88 722
89 730
90 731
91 751
92 786
93 787
94 799
95 800
96 815
97 816
98 817
99 830
100 831
101 832
102 857
103 876
104 883
105 898
106 906
107 907
108 909
109 911
110 916
111 917
112 918
113 924
114 939
115 950
116 985
117 996
118 998
119 1003
120 1027
121 1035
122 1036
123 1038
124 1043
125 1062
126 1078
127 1083
128 1085
129 1094
130 1101
131 1113
132 1153
133 1177
134 1183


In [702]:
f

3909

In [441]:
zz_threshed.shape

(100, 100)

## Step 2: make the nice UMAP image

In [ ]:
# TODO: use the animation pipeline to make a very small movie containing our desired figure
#       We can then extract one frame from that

In [708]:
zz_thresh_scale_matched = np.copy(zz_threshed)
zz_thresh_scale_matched[zz_thresh_scale_matched==1] = thresh

In [709]:
plt.figure()
plt.imshow(np.rot90(f), cmap='jet')
plt.colorbar()
plt.imshow(np.rot90(zz_thresh_scale_matched), alpha=0.4)
plt.text(15, 25, 'cluster 1', c='white')
plt.text(15, 85, 'cluster 2', c='white')
plt.text(40, 55, 'cluster 3', c='white')
plt.text(60, 85, 'cluster 4', c='white')
plt.text(80, 40, 'cluster 3', c='white')


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Text(80, 40, 'cluster 3')

In [710]:
def animate_still_movie(iteration):
    """
    Update the data held by the scatter plot and therefore animates it.
    """
    return


def make_still_umap_thresh_movie(f, zz_thresh_scale_matched, numFrames, savePath='/home/liam/temp/testanimation.mp4'):
    """

    Args:
    For f: see the thresholding section further up the notebook

    """
    plt.ioff()
    
    fig = plt.figure()
    plt.imshow(np.rot90(f), cmap='jet')
    plt.colorbar()
    plt.imshow(np.rot90(zz_thresh_scale_matched), alpha=0.4)
    plt.text(15, 25, 'cluster 0', c='white')
    plt.text(15, 85, 'cluster 1', c='white')
    plt.text(40, 55, 'cluster 2', c='white')
    plt.text(60, 85, 'cluster 3', c='white')
    plt.text(80, 40, 'cluster 4', c='white')

    ani = animation.FuncAnimation(fig, animate_still_movie, numFrames, fargs=(),
                                       interval=10, blit=False, repeat=True)

    Writer = animation.writers['ffmpeg']
    writer = Writer(fps=100, metadata=dict(artist='LOS'), bitrate=6000)
    ani.save(savePath, writer=writer)
    plt.ion()
    return

In [711]:
# Make cluster 1 movie

t0 = time.time()

savePath = '/home/liam/temp/umap_thresh_mov.mp4'
make_still_umap_thresh_movie(f, zz_thresh_scale_matched, 10, savePath=savePath)
      


print('Finished!')
tE = time.time()
print(tE-t0)

Finished!
1.5109648704528809


## Step 3: combine the movies

In [ ]:
# TODO: decide on the total number of frames the movie will be. The smallest of the 5, less the last model?
#
#       figure out the concatenations that you want

In [543]:
len(cluster_model_idxs)

for i in cluster_model_idxs:
    print(len(i))

370
537
76
17
31


In [525]:
# find the total number of frames in each cluster type
cluster_nframes = []

for clusterIdx in range(numClusters):
    
    cls_mod_idxs = cluster_model_idxs[clusterIdx]
    cls_mod_franges = all_windows[cls_mod_idxs]
    total_number_of_frames = np.sum(np.diff(cls_mod_franges,n=1,axis=1)[:,0])
    cluster_nframes.append(total_number_of_frames)

In [526]:
cluster_nframes

[13756, 14673, 2088, 472, 1491]

In [647]:
# let's just stack cluster idxs to generate loads of frames

cluster_model_idxs_edited = list(cluster_model_idxs)
cluster_model_idxs_edited[3] = cluster_model_idxs_edited[3]*10
cluster_model_idxs_edited[4] = cluster_model_idxs_edited[4]*4
cluster_model_idxs_edited[2] = cluster_model_idxs_edited[2]*2

# find the total number of frames in each cluster type
cluster_nframes_edited = []

for clusterIdx in range(numClusters):
    
    cls_mod_idxs = cluster_model_idxs_edited[clusterIdx]
    cls_mod_franges = all_windows[cls_mod_idxs]
    total_number_of_frames = np.sum(np.diff(cls_mod_franges,n=1,axis=1)[:,0])
    cluster_nframes_edited.append(total_number_of_frames)
    
cluster_nframes_edited

[13756, 14673, 4176, 4720, 5964]

In [649]:
numMovieFrames = 4176

In [650]:
cluster1_fld = '/home/liam/temp/clusters/1/'
cluster2_fld = '/home/liam/temp/clusters/2/'
cluster3_fld = '/home/liam/temp/clusters/3/'
cluster4_fld = '/home/liam/temp/clusters/4/'
cluster5_fld = '/home/liam/temp/clusters/5/'

cluster_flds = [cluster1_fld, cluster2_fld, cluster3_fld, cluster4_fld, cluster5_fld]

In [653]:
cls_mod_idxs = cluster_model_idxs_edited[0][:106]
cls_mod_franges = all_windows[cls_mod_idxs]
total_number_of_frames = np.sum(np.diff(cls_mod_franges,n=1,axis=1)[:,0])
total_number_of_frames

3913

In [654]:
cls_mod_idxs = cluster_model_idxs_edited[1][:2*80]
cls_mod_franges = all_windows[cls_mod_idxs]
total_number_of_frames = np.sum(np.diff(cls_mod_franges,n=1,axis=1)[:,0])
total_number_of_frames

4338

In [556]:
cls_mod_idxs = cluster_model_idxs_edited[2][:2*80]
cls_mod_franges = all_windows[cls_mod_idxs]
total_number_of_frames = np.sum(np.diff(cls_mod_franges,n=1,axis=1)[:,0])
total_number_of_frames

2088

In [565]:
cls_mod_idxs = cluster_model_idxs_edited[3][:2*76]
cls_mod_franges = all_windows[cls_mod_idxs]
total_number_of_frames = np.sum(np.diff(cls_mod_franges,n=1,axis=1)[:,0])
total_number_of_frames

2104

In [655]:
cls_mod_idxs = cluster_model_idxs_edited[4][:2*76]
cls_mod_franges = all_windows[cls_mod_idxs]
total_number_of_frames = np.sum(np.diff(cls_mod_franges,n=1,axis=1)[:,0])
total_number_of_frames

5964

In [656]:
# so in each cluster we should read 53, 80, 80, 76, 76 models
num_models_to_load = [2*53, 2*80, 2*80, 2*76, 2*76]

In [712]:
# grab the single umap frame

movpath = '/home/liam/temp/umap_thresh_mov.mp4'
cap = cv2.VideoCapture(movpath)
if cap.isOpened():
    ret, umap_frame = cap.read()
#print(umap_frame.shape)
cap.release()

#umap_frame = umap_frame[50:430, 100:570]
#umap_frame.shape

width = 600
height = 200
dim = (width, height)
umap_resized = cv2.resize(umap_frame[50:430, :], dim, interpolation = cv2.INTER_AREA) 

In [713]:
plt.figure()
plt.imshow(umap_resized)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [ ]:
resized.shape

In [661]:
# make movies of the same length for each cluster


clusterIdx = 0
cls_num_mods_to_load = num_models_to_load[clusterIdx]

outpath = '/home/liam/temp/clusters/1/combination.mp4'
width = int(600)
height = int(200)
dim = (width, height)
fps = 100
fourcc = cv2.VideoWriter_fourcc('m','p','4','v')
out = cv2.VideoWriter(outpath, fourcc, fps, dim)

for modIdx in cluster_model_idxs_edited[clusterIdx][:cls_num_mods_to_load]:
    mod_numframes = int(np.diff(all_windows[modIdx])[0])
    movpath = os.path.join(cluster_flds[clusterIdx], str(modIdx)+'.mp4')
    cap = cv2.VideoCapture(movpath)
    for f in range(mod_numframes):
        if cap.isOpened():
            ret, frame = cap.read()
            resized = cv2.resize(frame[:, 53:943], dim, interpolation = cv2.INTER_AREA) 
            # write the frame
            out.write(resized)

# close the writer
out.release()

In [662]:
# make movies of the same length for each cluster


clusterIdx = 1
cls_num_mods_to_load = num_models_to_load[clusterIdx]

outpath = '/home/liam/temp/clusters/2/combination.mp4'
width = int(600)
height = int(200)
dim = (width, height)
fps = 100
fourcc = cv2.VideoWriter_fourcc('m','p','4','v')
out = cv2.VideoWriter(outpath, fourcc, fps, dim)

for modIdx in cluster_model_idxs_edited[clusterIdx][:cls_num_mods_to_load]:
    mod_numframes = int(np.diff(all_windows[modIdx])[0])
    movpath = os.path.join(cluster_flds[clusterIdx], str(modIdx)+'.mp4')
    cap = cv2.VideoCapture(movpath)
    for f in range(mod_numframes):
        if cap.isOpened():
            ret, frame = cap.read()
            resized = cv2.resize(frame[:, 53:943], dim, interpolation = cv2.INTER_AREA) 
            # write the frame
            out.write(resized)

# close the writer
out.release()

In [663]:
# make movies of the same length for each cluster


clusterIdx = 2
cls_num_mods_to_load = num_models_to_load[clusterIdx]

outpath = '/home/liam/temp/clusters/3/combination.mp4'
width = int(600)
height = int(200)
dim = (width, height)
fps = 100
fourcc = cv2.VideoWriter_fourcc('m','p','4','v')
out = cv2.VideoWriter(outpath, fourcc, fps, dim)

for modIdx in cluster_model_idxs_edited[clusterIdx][:cls_num_mods_to_load]:
    mod_numframes = int(np.diff(all_windows[modIdx])[0])
    movpath = os.path.join(cluster_flds[clusterIdx], str(modIdx)+'.mp4')
    cap = cv2.VideoCapture(movpath)
    for f in range(mod_numframes):
        if cap.isOpened():
            ret, frame = cap.read()
            resized = cv2.resize(frame[:, 53:943], dim, interpolation = cv2.INTER_AREA) 
            # write the frame
            out.write(resized)

# close the writer
out.release()

In [664]:
# make movies of the same length for each cluster


clusterIdx = 3
cls_num_mods_to_load = num_models_to_load[clusterIdx]

outpath = '/home/liam/temp/clusters/4/combination.mp4'
width = int(600)
height = int(200)
dim = (width, height)
fps = 100
fourcc = cv2.VideoWriter_fourcc('m','p','4','v')
out = cv2.VideoWriter(outpath, fourcc, fps, dim)

for modIdx in cluster_model_idxs_edited[clusterIdx][:cls_num_mods_to_load]:
    mod_numframes = int(np.diff(all_windows[modIdx])[0])
    movpath = os.path.join(cluster_flds[clusterIdx], str(modIdx)+'.mp4')
    cap = cv2.VideoCapture(movpath)
    for f in range(mod_numframes):
        if cap.isOpened():
            ret, frame = cap.read()
            resized = cv2.resize(frame[:, 53:943], dim, interpolation = cv2.INTER_AREA) 
            # write the frame
            out.write(resized)

# close the writer
out.release()

In [665]:
# make movies of the same length for each cluster


clusterIdx = 4
cls_num_mods_to_load = num_models_to_load[clusterIdx]

outpath = '/home/liam/temp/clusters/5/combination.mp4'
width = int(600)
height = int(200)
dim = (width, height)
fps = 100
fourcc = cv2.VideoWriter_fourcc('m','p','4','v')
out = cv2.VideoWriter(outpath, fourcc, fps, dim)

for modIdx in cluster_model_idxs_edited[clusterIdx][:cls_num_mods_to_load]:
    mod_numframes = int(np.diff(all_windows[modIdx])[0])
    movpath = os.path.join(cluster_flds[clusterIdx], str(modIdx)+'.mp4')
    cap = cv2.VideoCapture(movpath)
    for f in range(mod_numframes):
        if cap.isOpened():
            ret, frame = cap.read()
            resized = cv2.resize(frame[:, 53:943], dim, interpolation = cv2.INTER_AREA) 
            # write the frame
            out.write(resized)

# close the writer
out.release()

In [669]:
cluster_mov_paths = ['/home/liam/temp/clusters/1/combination.mp4','/home/liam/temp/clusters/2/combination.mp4','/home/liam/temp/clusters/3/combination.mp4',
                    '/home/liam/temp/clusters/4/combination.mp4', '/home/liam/temp/clusters/5/combination.mp4']


In [715]:
# make the master movie
outpath = '/home/liam/temp/clusters/movie.mp4'
width = int(2*600)
height = int(3*200)
dim = (width, height)
fps = 100
fourcc = cv2.VideoWriter_fourcc('m','p','4','v')
out = cv2.VideoWriter(outpath, fourcc, fps, dim)

caps = []
for i in range(numClusters):
    caps.append(cv2.VideoCapture(cluster_mov_paths[i]))
    
for f in range(3910):
    #print(f)
    
    # load the 5 movie frames
    if caps[0].isOpened():
        ret, frame1 = caps[0].read()
    if caps[1].isOpened():
        ret, frame2 = caps[1].read() 
    if caps[2].isOpened():
        ret, frame3 = caps[2].read()
    if caps[3].isOpened():
        ret, frame4 = caps[3].read()
    if caps[4].isOpened():
        ret, frame5 = caps[4].read()
        
    # combine everthing together
    top_row = np.hstack([frame1, frame2])
    mid_row = np.hstack([frame3, frame4])
    bottom_row = np.hstack([frame5, umap_resized])
    movieFrame = np.vstack([top_row, mid_row, bottom_row])  
    
    # write the frame
    out.write(movieFrame)


# go for 40 secs = 4000 frames
out.release()


In [696]:
movieFrame.shape

(600, 1200, 3)

In [697]:
dim

(1200, 600)

In [675]:
top_row.shape

(200, 1200, 3)

In [676]:
umap_frame.shape

(480, 640, 3)

In [716]:
plt.figure()
plt.imshow(movieFrame)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

# grab plane individual frames

In [27]:
video_paths = [xzMovPath, xyMovPath, yzMovPath]
caps = [cv2.VideoCapture(video_path.value) for video_path in video_paths]
[cap.set(1,local_index.value) for cap in caps] #go to current frame
# grab the frame from each camera
if caps[0].isOpened() and caps[1].isOpened() and caps[2].isOpened():  
    # grab the 3 images
    ret_xz, frame_xz = caps[0].read()
    if not ret_xz:
        raise TypeError('xz movie for splitdata_idx={0} not opening'.format(splitdata_idx.value))
    ret_xy, frame_xy = caps[1].read()
    if not ret_xy:
        raise TypeError('xy movie for splitdata_idx={0} not opening'.format(splitdata_idx.value))
    ret_yz, frame_yz = caps[2].read()
    if not ret_yz:
        raise TypeError('yz movie for splitdata_idx={0} not opening'.format(splitdata_idx.value))
    frames = [frame_xz, frame_xy, frame_yz]
    
plt.ion()
plt.figure()
plt.imshow(frames[0])

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [30]:
plt.imsave('/home/liam/Dropbox/transfer/xz_frame.tiff', frames[0])
plt.imsave('/home/liam/Dropbox/transfer/xy_frame.tiff', frames[1])
plt.imsave('/home/liam/Dropbox/transfer/yz_frame.tiff', frames[2])